In [417]:
import pandas as pd
import os
import json
import random
import numpy as np
import warnings

with warnings.catch_warnings():
    warnings.simplefilter("ignore")

In [419]:
cwd = os.getcwd()

train = pd.read_parquet(f'{cwd}\\Vulnerable-Verified-Smart-Contracts\\train.parquet')
test = pd.read_parquet(f'{cwd}\\Vulnerable-Verified-Smart-Contracts\\test.parquet')
validation = pd.read_parquet(f'{cwd}\\Vulnerable-Verified-Smart-Contracts\\validation.parquet')

In [420]:
print("train:", train.columns.tolist())

train: ['contract_name', 'file_path', 'contract_address', 'language', 'source_code', 'overlapping', 'soldetector', 'slither', 'oyente', 'smartcheck', 'compiler_version', 'license_type', 'swarm_source']


In [421]:
random_row = random.randint(0, len(train) - 1)  # Generates a number between 0 and (len(train) - 1) (inclusive)
print("Random row:", random_row)

source_code = train.iloc[random_row]['source_code']
overlapping = train.iloc[random_row]['overlapping']
slither = train.iloc[random_row]['slither']
print("\nsource code: \n" + source_code + "\n" + "\noverlapping: \n" + overlapping + "\nslither: \n" + slither)

Random row: 0

source code: 
pragma solidity ^0.4.22;

contract Gift
{
    bool closed = false;
    uint unlockTime = 43200;
    address sender;
    address receiver;
 
    function Put(address _receiver) public payable {
        if ((!closed && msg.value > 0.5 ether) || sender == 0x0 ) {
            sender = msg.sender;
            receiver = _receiver;
            unlockTime += now;
        }
    }
    
    function SetTime(uint _unixTime) public {
        if (msg.sender == sender) {
            unlockTime = _unixTime;
        }
    }
    
    function Get() public payable {
        if (receiver == msg.sender && now >= unlockTime) {
            msg.sender.transfer(address(this).balance);
        }
    }
    
    function Close() public {
        if (sender == msg.sender) {
           closed=true;
        }
    }

    function() public payable { }
}

overlapping: 
{"IOU": [[{"defect": "Integer_overflow_and_underflow", "lines": [14], "severity": "High", "tool": "soldetector", "type": "

In [425]:
overlapping = train.iloc[random_row]['overlapping']

# Parse JSON string to Python dictionary
data = json.loads(overlapping)

# Output the dictionary
print(data)
print(type(data))  # <class 'dict'>
print(data.keys())

{'IOU': [[{'defect': 'Integer_overflow_and_underflow', 'lines': [14], 'severity': 'High', 'tool': 'soldetector', 'type': 'Code_specification'}, {'error': 'Integer Overflow.', 'level': 'Warning', 'line': 14, 'tool': 'oyente'}]], 'TD': [[{'defect': 'Dependency_of_timestamp', 'lines': [25], 'severity': 'Medium', 'tool': 'soldetector', 'type': 'Business_logic'}, {'check': 'timestamp', 'confidence': 'Medium', 'impact': 'Low', 'lines': [25], 'tool': 'slither'}, {'error': 'Timestamp Dependency.', 'level': 'Warning', 'line': 25, 'tool': 'oyente'}]]}
<class 'dict'>
dict_keys(['IOU', 'TD'])


In [426]:
def check_vuln_types(dataset):
    count = 0
    vulnerability_keys = set()
    
    for index, row in dataset.iterrows():
        overlapping = row['overlapping']
    
        # Parse JSON string to Python dictionary
        data = json.loads(overlapping)
    
        vuln = data.keys()
        for key in vuln:
            vulnerability_keys.add(key)
        count += len(vuln)
    return str(vulnerability_keys) + ", " + str(len(vulnerability_keys)) + " keys, total vulnerabilities: " + str(count)

print("train", check_vuln_types(train))
print("test", check_vuln_types(test))
print("valid", check_vuln_types(validation))

train {'FE', 'TOD', 'TO', 'UcC', 'IOU', 'RE', 'UpS', 'TD', 'NC', 'DC'}, 10 keys, total vulnerabilities: 710
test {'TOD', 'UcC', 'IOU', 'RE', 'UpS', 'TD', 'NC', 'DC'}, 8 keys, total vulnerabilities: 46
valid {'TO', 'TOD', 'UcC', 'IOU', 'RE', 'UpS', 'TD', 'NC', 'DC'}, 9 keys, total vulnerabilities: 66


In [123]:
def add_new_columns(dataset):
    vulns = ['IOU', 'TOD', 'RE', 'UcC', 'UpS', 'TD', 'TO', 'DC', 'NC', 'FE']
    tools = ['soldetector', 'slither', 'oyente', 'smartcheck']

    # Initialize new columns
    for vuln in vulns:
        dataset[f'{vuln}'] = "Undetected"
        for tool in tools:
            dataset[f'{vuln}_{tool}'] = "Undetected"
            
    for index, row in dataset.iterrows():
        overlapping = row['overlapping']
    
        # Parse JSON string to Python dictionary
        data = json.loads(overlapping)
    
        vulns = data.keys()
        for vuln in vulns:
            info_dict_list = data[vuln][0]
            dataset.at[index, f'{vuln}'] = "Detected"
            for info_dict in info_dict_list:
                # print(info_dict)
                try:
                    lines = str(info_dict['lines'])
                except KeyError as e:
                    lines = "[" + str(info_dict['line']) + "]"
                tool = info_dict['tool']
                dataset.at[index, f'{vuln}_{tool}'] = lines
add_new_columns(test)

In [125]:
test.columns

Index(['contract_name', 'file_path', 'contract_address', 'language',
       'source_code', 'overlapping', 'soldetector', 'slither', 'oyente',
       'smartcheck', 'compiler_version', 'license_type', 'swarm_source', 'IOU',
       'IOU_soldetector', 'IOU_slither', 'IOU_oyente', 'IOU_smartcheck', 'TOD',
       'TOD_soldetector', 'TOD_slither', 'TOD_oyente', 'TOD_smartcheck', 'RE',
       'RE_soldetector', 'RE_slither', 'RE_oyente', 'RE_smartcheck', 'UcC',
       'UcC_soldetector', 'UcC_slither', 'UcC_oyente', 'UcC_smartcheck', 'UpS',
       'UpS_soldetector', 'UpS_slither', 'UpS_oyente', 'UpS_smartcheck', 'TD',
       'TD_soldetector', 'TD_slither', 'TD_oyente', 'TD_smartcheck', 'TO',
       'TO_soldetector', 'TO_slither', 'TO_oyente', 'TO_smartcheck', 'DC',
       'DC_soldetector', 'DC_slither', 'DC_oyente', 'DC_smartcheck', 'NC',
       'NC_soldetector', 'NC_slither', 'NC_oyente', 'NC_smartcheck', 'FE',
       'FE_soldetector', 'FE_slither', 'FE_oyente', 'FE_smartcheck'],
      dtype='o

In [127]:
test

,contract_name,file_path,contract_address,language,source_code,overlapping,soldetector,slither,oyente,smartcheck,...,NC,NC_soldetector,NC_slither,NC_oyente,NC_smartcheck,FE,FE_soldetector,FE_slither,FE_oyente,FE_smartcheck
0,CrystalReignShard,CrystalReignShard.sol,0x5108e432d40d34a4d0ed88c12e83db0e571bfc75,Solidity,pragma solidity ^0.4.4;\r\n\r\n\r\n/**\r\n* @t...,"{""IOU"": [[{""defect"": ""Integer_overflow_and_und...","[{""defect"": ""Leaking_to_arbitary_address"", ""ty...","[{""check"": ""constable-states"", ""impact"": ""Opti...","[{""error"": ""Integer Underflow."", ""line"": 147, ...","[{""rule"": ""SOLIDITY_ADDRESS_HARDCODED"", ""line""...",...,Detected,[220],[221],Undetected,[220],Undetected,Undetected,Undetected,Undetected,Undetected
1,WrapperLockEth,WrapperLockEth.sol,0x9fb2fd4bce3061b826bf1a0b53b737c6f3e0e586,Solidity,pragma solidity ^0.4.22;\r\n\r\n\r\n\r\n/**\r\...,"{""TD"": [[{""defect"": ""Dependency_of_timestamp"",...","[{""defect"": ""Erroneous_constructor_name"", ""typ...","[{""check"": ""constable-states"", ""impact"": ""Opti...","[{""error"": ""Integer Overflow."", ""line"": 114, ""...","[{""rule"": ""SOLIDITY_ADDRESS_HARDCODED"", ""line""...",...,Undetected,Undetected,Undetected,Undetected,Undetected,Undetected,Undetected,Undetected,Undetected,Undetected
2,GetsBurned,GetsBurned.sol,0x5f15d2d4c60e229586a5bdfe2eec1981f92845c1,Solidity,pragma solidity ^0.4.25; \r\n\r\ncontract Get...,"{""UpS"": [[{""defect"": ""Unprotected_Suicide"", ""l...","[{""defect"": ""Unprotected_Suicide"", ""type"": ""Fu...","[{""check"": ""external-function"", ""impact"": ""Opt...",[],"[{""rule"": ""SOLIDITY_PRAGMAS_VERSION"", ""line"": ...",...,Undetected,Undetected,Undetected,Undetected,Undetected,Undetected,Undetected,Undetected,Undetected,Undetected
3,guess_tw,guess_tw.sol,0xa9aebbf67433e3e8206af6fc2ddd99ff8e7cc137,Solidity,pragma solidity ^0.4.20;\r\n\r\ncontract guess...,"{""TOD"": [[{""defect"": ""Transaction_order_depend...","[{""defect"": ""Transaction_order_dependency"", ""t...","[{""check"": ""external-function"", ""impact"": ""Opt...","[{""error"": ""Integer Underflow."", ""line"": 11, ""...","[{""rule"": ""SOLIDITY_PRAGMAS_VERSION"", ""line"": ...",...,Undetected,Undetected,Undetected,Undetected,Undetected,Undetected,Undetected,Undetected,Undetected,Undetected
4,WithdrawContract,WithdrawContract.sol,0x0efef0b34638addc8833ba729ae20016b5f24ffc,Solidity,//File: node_modules/giveth-common-contracts/c...,"{""IOU"": [[{""defect"": ""Integer_overflow_and_und...","[{""defect"": ""Leaking_to_arbitary_address"", ""ty...","[{""check"": ""arbitrary-send"", ""impact"": ""High"",...","[{""error"": ""Transaction-Ordering Dependency."",...","[{""rule"": ""SOLIDITY_ADDRESS_HARDCODED"", ""line""...",...,Undetected,Undetected,Undetected,Undetected,Undetected,Undetected,Undetected,Undetected,Undetected,Undetected
5,we_play,we_play.sol,0x6df766e4b524aa1d4b4b9405994ca69dc161da3f,Solidity,pragma solidity ^0.4.25;\r\n\r\ncontract we_pl...,"{""TOD"": [[{""defect"": ""Transaction_order_depend...","[{""defect"": ""Transaction_order_dependency"", ""t...","[{""check"": ""external-function"", ""impact"": ""Opt...","[{""error"": ""Integer Underflow."", ""line"": 8, ""l...","[{""rule"": ""SOLIDITY_ADDRESS_HARDCODED"", ""line""...",...,Undetected,Undetected,Undetected,Undetected,Undetected,Undetected,Undetected,Undetected,Undetected,Undetected
6,EthCapsule,EthCapsule.sol,0x5274a2293dbe075a82d41e873bb927403a9dce46,Solidity,pragma solidity ^0.4.11;\r\n\r\ncontract Ownab...,"{""IOU"": [[{""defect"": ""Integer_overflow_and_und...","[{""defect"": ""Leaking_to_arbitary_address"", ""ty...","[{""check"": ""external-function"", ""impact"": ""Opt...","[{""error"": ""Integer Overflow."", ""line"": 107, ""...","[{""rule"": ""SOLIDITY_DEPRECATED_CONSTRUCTIONS"",...",...,Undetected,Undetected,Undetected,Undetected,Undetected,Undetected,Undetected,Undetected,Undetected,Undetected
7,PonziToken,PonziToken.sol,0x33f00114d5aca3dee03189d4ed9d4f886dad84b0,Solidity,pragma so

In [129]:
# Now we need to check entries that have unequal line numbers marked as vulnerable by different tools
# First let's check for Reentrancy vulnerability
def check_for_improper_detections(dataset):
    vulns = ['IOU', 'TOD', 'RE', 'UcC', 'UpS', 'TD', 'TO', 'DC', 'NC', 'FE']
    tools = ['soldetector', 'slither', 'oyente', 'smartcheck']
    result = dict()
    
    for vuln in vulns:
        vuln_df = dataset[dataset[vuln] == "Detected"]

        if len(vuln_df) > 0:
            print("Vulnerability: ", vuln, "Number of samples: ", len(vuln_df))
            df = pd.DataFrame(columns=["file_path", "contract_address", "soldetector", "slither", "oyente", "smartcheck", "need_intervention"])
            # Set all columns except 'need_intervention' to 'object' type
            df[df.columns.difference(['need_intervention'])] = df[df.columns.difference(['need_intervention'])].astype('object')
            
            row_index = 0
            for index, row in vuln_df.iterrows():
                df.loc[row_index] = [row['file_path'], row['contract_address'], None, None, None, None, False]
                vul_lines_set_for_all_tool = []
                need_intervention = False
                for tool in tools:
                    vul_lines_set_for_tool = set()
                    if row[f'{vuln}_{tool}'] != "Undetected":
                        line_list = eval(row[f'{vuln}_{tool}'])
                        vul_lines_set_for_tool.update(line_list)
                        vul_lines_set_for_all_tool.append(vul_lines_set_for_tool)
                        if len(vul_lines_set_for_tool) > 0:
                            df.loc[row_index, tool] = str(vul_lines_set_for_tool)
                for i in range(len(vul_lines_set_for_all_tool)):
                    for j in range(len(vul_lines_set_for_all_tool)):
                        if i == j:
                            continue
                        if vul_lines_set_for_all_tool[i] != vul_lines_set_for_all_tool[j]:
                            need_intervention = True
                df.loc[row_index, "need_intervention"] = need_intervention
                row_index += 1
            result[vuln] = df
    return result

result_dict = check_for_improper_detections(test)      

Vulnerability:  IOU Number of samples:  11
Vulnerability:  TOD Number of samples:  12
Vulnerability:  RE Number of samples:  2
Vulnerability:  UcC Number of samples:  1
Vulnerability:  UpS Number of samples:  4
Vulnerability:  TD Number of samples:  8
Vulnerability:  DC Number of samples:  1
Vulnerability:  NC Number of samples:  7


In [131]:
def count_need_intervention(result_dict):
    print("Vulberability types: ", list(result_dict.keys()))
    print("Number of vulnerability types: ", len(result_dict.keys()))
    count_dict = dict()
    count_sum = 0
    for key in result_dict.keys():
        df = result_dict[key]
        true_count = (df['need_intervention'] == True).sum()
        count_sum += true_count
        count_dict[key] = true_count
    return count_sum, count_dict

count, count_dict = count_need_intervention(result_dict)
print("Number of contracts needing intervention: ", count)
print("Number of contracts needing intervention by vuln. type: ", count_dict)
display(result_dict['DC'])

Vulberability types:  ['IOU', 'TOD', 'RE', 'UcC', 'UpS', 'TD', 'DC', 'NC']
Number of vulnerability types:  8
Number of contracts needing intervention:  16
Number of contracts needing intervention by vuln. type:  {'IOU': 2, 'TOD': 0, 'RE': 2, 'UcC': 0, 'UpS': 3, 'TD': 1, 'DC': 1, 'NC': 7}


,file_path,contract_address,soldetector,slither,oyente,smartcheck,need_intervention
0,Pair.sol,0x0b3ded1b54e89f8899482cfccf515d525835bc2b,{472},"{472, 473}",None,None,True


# Finding the smart contracts that need intervention across all the datasets(train, validation, test)

## Vulnerabilities

- **DC (DelegateCall)** : The address.delegatecall() function allows a smart contract to dynamically load external contracts from address at runtime. If the attacker can control the external contract and affect the current contract status, the contract is vulnerable to DC.
- **IOU (Arithmetic/Integer Overflow and Underflow)**: An arithmetic overflow or underflow, often called Integer Overflow or Underflow (IOU), occurs when an arithmetic operation attempts to create a numeric variable value that is larger than the maximum value or smaller than the minimum value of the variable type. If the arithmetic operation may pass a variable type’s maximum or minimum value and  is performed without using SafeMath, the contract is vulnerable to IOU.
- **NC (Nested Call)**: The function containing the loop has a high risk of exceeding its gas limitation and causing an out-of-gas error. If the attacker can control the loop iteration and causes the out-of-gas error, the contract is vulnerable to NC.
- **RE (Reentrancy)**: The contract vulnerable to RE uses the call() function to transfer ether to an external contract. The external contract can reenter the vulnerable contract by fallback function. If the state variable change is after the call() function, the reentrance will cause status inconsistency
- **TD (Timestamp Dependency)**: The contract uses the timestamp as the deciding factor for critical operations, e.g., sending ether. If the attacker can get ether from the contract by manipulating the timestamp or affecting the critical operations, the contract is vulnerable to TD
- **TO (TxOrigin)**: If the contract only uses tx.origin to verify the caller's identification for critical operations, it is vulnerable to TO
- **TOD (Transaction Order Dependency)**: The contract may send out ether differently according to different values of a global state variable or different balance values of the contract. If the attackers can get ether from the contract by manipulating the transaction sequences, the contract is vulnerable to TOD.
- **UcC (Unchecked Call)**: The contract uses the function call() or send() without result checking. If the send() or call()  function fails and leads to status inconsistency, the contract is vulnerable to UcC.
- **UpS(Unprotected Suicide)**: If an attacker can self-destruct the contract by calling the selfdestruct(address) function, the contract is vulnerable to UpS.
- **FE (Frozen Ether)**:  the contract can receive ether but cannot transfer it by itself, it is vulnerable to FE.

In [47]:
import pandas as pd
import pickle
import os
import json
import random
import numpy as np
import anthropic
import warnings
import time

with warnings.catch_warnings():
    warnings.simplefilter("ignore")

In [49]:
cwd = os.getcwd()

train = pd.read_parquet(f'{cwd}\\Vulnerable-Verified-Smart-Contracts\\train.parquet')
test = pd.read_parquet(f'{cwd}\\Vulnerable-Verified-Smart-Contracts\\test.parquet')
validation = pd.read_parquet(f'{cwd}\\Vulnerable-Verified-Smart-Contracts\\validation.parquet')

## Helper functions

In [52]:
def add_new_columns(dataset):
    vulns = ['IOU', 'TOD', 'RE', 'UcC', 'UpS', 'TD', 'TO', 'DC', 'NC', 'FE']
    tools = ['soldetector', 'slither', 'oyente', 'smartcheck']

    # Initialize new columns
    for vuln in vulns:
        dataset[f'{vuln}'] = "Undetected"
        for tool in tools:
            dataset[f'{vuln}_{tool}'] = "Undetected"
            
    for index, row in dataset.iterrows():
        overlapping = row['overlapping']
    
        # Parse JSON string to Python dictionary
        data = json.loads(overlapping)
    
        vulns = data.keys()
        for vuln in vulns:
            info_dict_list = data[vuln][0]
            dataset.at[index, f'{vuln}'] = "Detected"
            for info_dict in info_dict_list:
                # print(info_dict)
                try:
                    lines = str(info_dict['lines'])
                except KeyError as e:
                    lines = "[" + str(info_dict['line']) + "]"
                tool = info_dict['tool']
                dataset.at[index, f'{vuln}_{tool}'] = lines

def check_for_improper_detections(dataset):
    vulns = ['IOU', 'TOD', 'RE', 'UcC', 'UpS', 'TD', 'TO', 'DC', 'NC', 'FE']
    tools = ['soldetector', 'slither', 'oyente', 'smartcheck']
    result = dict()
    
    for vuln in vulns:
        vuln_df = dataset[dataset[vuln] == "Detected"]

        if len(vuln_df) > 0:
            # print("Vulnerability: ", vuln, "Number of samples: ", len(vuln_df))
            df = pd.DataFrame(columns=["file_path", "contract_address", "soldetector", "slither", "oyente", "smartcheck", "need_intervention"])
            # Set all columns except 'need_intervention' to 'object' type
            df[df.columns.difference(['need_intervention'])] = df[df.columns.difference(['need_intervention'])].astype('object')
            
            row_index = 0
            for index, row in vuln_df.iterrows():
                df.loc[row_index] = [row['file_path'], row['contract_address'], None, None, None, None, False]
                vul_lines_set_for_all_tool = []
                need_intervention = False
                for tool in tools:
                    vul_lines_set_for_tool = set()
                    if row[f'{vuln}_{tool}'] != "Undetected":
                        line_list = eval(row[f'{vuln}_{tool}'])
                        vul_lines_set_for_tool.update(line_list)
                        vul_lines_set_for_all_tool.append(vul_lines_set_for_tool)
                        if len(vul_lines_set_for_tool) > 0:
                            df.loc[row_index, tool] = str(vul_lines_set_for_tool)
                for i in range(len(vul_lines_set_for_all_tool)):
                    for j in range(len(vul_lines_set_for_all_tool)):
                        if i == j:
                            continue
                        if vul_lines_set_for_all_tool[i] != vul_lines_set_for_all_tool[j]:
                            need_intervention = True
                df.loc[row_index, "need_intervention"] = need_intervention
                row_index += 1
            result[vuln] = df
    return result

def count_need_intervention(result_dict):
    # print("Vulberability types: ", list(result_dict.keys()))
    # print("Number of vulnerability types: ", len(result_dict.keys()))
    count_dict = dict()
    count_sum = 0
    for key in result_dict.keys():
        df = result_dict[key]
        true_count = (df['need_intervention'] == True).sum()
        count_sum += true_count
        count_dict[key] = true_count
    return count_sum, count_dict

In [54]:
# Add new columns to the dataframes for further processing
add_new_columns(train)
add_new_columns(test)
add_new_columns(validation)

In [56]:
result_dict_train = check_for_improper_detections(train)
result_dict_test = check_for_improper_detections(test)
result_dict_validation = check_for_improper_detections(validation)

count_train, count_dict_train = count_need_intervention(result_dict_train)
count_test, count_dict_test = count_need_intervention(result_dict_test)
count_validation, count_dict_validation = count_need_intervention(result_dict_validation)

print("Train dataset\n===============================\n")
print("Number of contracts needing intervention: ", count_train)
print("Number of contracts needing intervention by vuln. type: ", count_dict_train)

print("\n\nTest dataset\n===============================\n")
print("Number of contracts needing intervention: ", count_test)
print("Number of contracts needing intervention by vuln. type: ", count_dict_test)

print("\n\nValidation dataset\n===============================")
print("Number of contracts needing intervention: ", count_validation)
print("Number of contracts needing intervention by vuln. type: ", count_dict_validation)

Train dataset

Number of contracts needing intervention:  267
Number of contracts needing intervention by vuln. type:  {'IOU': 30, 'TOD': 1, 'RE': 56, 'UcC': 0, 'UpS': 64, 'TD': 5, 'TO': 0, 'DC': 9, 'NC': 100, 'FE': 2}


Test dataset

Number of contracts needing intervention:  16
Number of contracts needing intervention by vuln. type:  {'IOU': 2, 'TOD': 0, 'RE': 2, 'UcC': 0, 'UpS': 3, 'TD': 1, 'DC': 1, 'NC': 7}


Validation dataset
Number of contracts needing intervention:  18
Number of contracts needing intervention by vuln. type:  {'IOU': 0, 'TOD': 0, 'RE': 1, 'UcC': 0, 'UpS': 5, 'TD': 0, 'TO': 0, 'DC': 1, 'NC': 11}


## Check one example to build a prompt suitable for narrowing the vulnerable line with the help of a LLM

In [272]:
df = result_dict_train['RE']

In [15]:
example_address = "0xdd71e35f680bb5adc77c6d1d9ef5793598e613dc"
processed_row = df[df['contract_address'] == example_address].iloc[0]
original_row = train[train['contract_address'] == example_address]
print("Information\n======================")
soldetector_lines = processed_row['soldetector']
slither_lines = processed_row['slither']
oyente_lines = processed_row['oyente']
smartcheck_lines = processed_row['smartcheck']
print("soldetector_line: ", soldetector_lines)
print("slither_line: ", slither_lines)
print("oyente_line: ", oyente_lines)
print("smartcheck_line: ", smartcheck_lines)


print("\n\nSource code\n====================")
source_code_value = original_row.iloc[0]['source_code']
print(source_code_value)

print("\n\nLine extraction\n====================")
if soldetector_lines != None:
    print("soldetector--------------------------------------------------")
    lines = source_code_value.splitlines()
    line_numbers = list(eval(soldetector_lines))
    line_numbers.sort()
    for line_number in line_numbers:
        l = lines[line_number - 1]
        print(l)

if slither_lines != None:
    print("slither--------------------------------------------------")
    lines = source_code_value.splitlines()
    line_numbers = list(eval(slither_lines))
    line_numbers.sort()
    for line_number in line_numbers:
        l = lines[line_number - 1]
        print(l)
        
if oyente_lines != None:
    print("oyente--------------------------------------------------")
    lines = source_code_value.splitlines()
    line_numbers = list(eval(oyente_lines))
    line_numbers.sort()
    for line_number in line_numbers:
        l = lines[line_number - 1]
        print(l)

if smartcheck_lines != None:
    print("smartcheck--------------------------------------------------")
    lines = source_code_value.splitlines()
    line_numbers = list(eval(smartcheck_lines))
    line_numbers.sort()
    for line_number in line_numbers:
        l = lines[line_number - 1]
        print(l)

Information
soldetector_line:  {22}
slither_line:  {24}
oyente_line:  {22}
smartcheck_line:  None


Source code
pragma solidity ^0.4.25;

contract Piggy_BanK
{
    function Put(uint _unlockTime)
    public
    payable
    {
        var acc = Acc[msg.sender];
        acc.balance += msg.value;
        acc.unlockTime = _unlockTime>now?_unlockTime:now;
        LogFile.AddMessage(msg.sender,msg.value,"Put");
    }

    function Collect(uint _am)
    public
    payable
    {
        var acc = Acc[msg.sender];
        if( acc.balance>=MinSum && acc.balance>=_am && now>acc.unlockTime)
        {
            if(msg.sender.call.value(_am)())
            {
                acc.balance-=_am;
                LogFile.AddMessage(msg.sender,_am,"Collect");
            }
        }
    }

    function() 
    public 
    payable
    {
        Put(0);
    }

    struct Holder   
    {
        uint unlockTime;
        uint balance;
    }

    mapping (address => Holder) public Acc;

    Log LogFile;

    uin

In [41]:
client = anthropic.Anthropic(
    # defaults to os.environ.get("ANTHROPIC_API_KEY")
    # api_key="",
)

In [43]:
client.models.list(limit=20)

SyncPage[ModelInfo](data=[ModelInfo(id='claude-3-7-sonnet-20250219', created_at=datetime.datetime(2025, 2, 24, 0, 0, tzinfo=datetime.timezone.utc), display_name='Claude 3.7 Sonnet', type='model'), ModelInfo(id='claude-3-5-sonnet-20241022', created_at=datetime.datetime(2024, 10, 22, 0, 0, tzinfo=datetime.timezone.utc), display_name='Claude 3.5 Sonnet (New)', type='model'), ModelInfo(id='claude-3-5-haiku-20241022', created_at=datetime.datetime(2024, 10, 22, 0, 0, tzinfo=datetime.timezone.utc), display_name='Claude 3.5 Haiku', type='model'), ModelInfo(id='claude-3-5-sonnet-20240620', created_at=datetime.datetime(2024, 6, 20, 0, 0, tzinfo=datetime.timezone.utc), display_name='Claude 3.5 Sonnet (Old)', type='model'), ModelInfo(id='claude-3-haiku-20240307', created_at=datetime.datetime(2024, 3, 7, 0, 0, tzinfo=datetime.timezone.utc), display_name='Claude 3 Haiku', type='model'), ModelInfo(id='claude-3-opus-20240229', created_at=datetime.datetime(2024, 2, 29, 0, 0, tzinfo=datetime.timezone.ut

In [45]:
message = client.messages.create(
    model="claude-3-7-sonnet-20250219",
    max_tokens=1024,
    messages=[
        {"role": "user", "content": "Hello, Claude"}
    ]
)
print(message.content)

[TextBlock(citations=None, text="Hello! It's nice to meet you. How can I help you today? I'm here to assist with information, answer questions, have a conversation, or help with various tasks. What would you like to discuss or explore?", type='text')]


In [60]:
prompt_part_1 = """
Analyze the smart contract for reentrancy vulnerabilities and identify which line or lines make the smart contract vulnerable to reentrancy attacks.  

Here's the scenario:

Contract code:
```
{contract_code}
```

"""
prompt_part_2 = """
Vulnerability detection tool #{tool_number} identified {this_line_or_these_lines} as vulnerable:
{line_or_lines}

"""

prompt_part_3 = """
Think before you give the final answer. First, think about what a reentrancy vulnerability is and what are the possible ways the reentrancy vulnerability can occur in a smart contract. Then, examine the smart contract code and the results of the vulnerability detection tools to guide your analysis of how reentrancy vulnerability can occur in the given smart contract, and identify the scenarios in which reentrancy vulnerability occurs. Then evaluate scenarios to verify they are correct. Then output only the line or lines that make the smart contract vulnerable to reentrancy attacks. 

Provide your response in exactly this format:
VULNERABLE_LINES: [List all the lines that create the reenctrancy vulnerability a possibility in the given smart contract]
"""

In [62]:
prompt = prompt_part_1.format(contract_code=source_code_value)
prompt += prompt_part_2.format(tool_number="1", this_line_or_these_lines="this line", line_or_lines="```if(msg.sender.call.value(_am)())```")
prompt += prompt_part_2.format(tool_number="2", this_line_or_these_lines="this line", line_or_lines="```acc.balance-=_am;```")
prompt += prompt_part_2.format(tool_number="3", this_line_or_these_lines="this line", line_or_lines="```if(msg.sender.call.value(_am)())```")
prompt += prompt_part_3

print(prompt)


Analyze the smart contract for reentrancy vulnerabilities and identify which line or lines make the smart contract vulnerable to reentrancy attacks.  

Here's the scenario:

Contract code:
```
pragma solidity ^0.4.25;

contract Piggy_BanK
{
    function Put(uint _unlockTime)
    public
    payable
    {
        var acc = Acc[msg.sender];
        acc.balance += msg.value;
        acc.unlockTime = _unlockTime>now?_unlockTime:now;
        LogFile.AddMessage(msg.sender,msg.value,"Put");
    }

    function Collect(uint _am)
    public
    payable
    {
        var acc = Acc[msg.sender];
        if( acc.balance>=MinSum && acc.balance>=_am && now>acc.unlockTime)
        {
            if(msg.sender.call.value(_am)())
            {
                acc.balance-=_am;
                LogFile.AddMessage(msg.sender,_am,"Collect");
            }
        }
    }

    function() 
    public 
    payable
    {
        Put(0);
    }

    struct Holder   
    {
        uint unlockTime;
        uint bala

In [64]:
response = client.messages.create(
    model="claude-3-7-sonnet-20250219",
    max_tokens=8000,
    system="You are a blockchain security expert specializing in smart contract vulnerability analysis. Your task is to analyze Solidity smart contracts for reentrancy vulnerabilities.",
    thinking={
        "type": "enabled",
        "budget_tokens": 4000
    },
    messages=[{
        "role": "user",
        "content": prompt
    }]
)

In [66]:
print(response.content[1].text)

I need to analyze this contract for reentrancy vulnerabilities by examining how it handles external calls and state updates.

A reentrancy vulnerability occurs when a contract:
1. Makes an external call to an untrusted address
2. Updates its state after that external call
3. Lacks protection against recursive calls

Looking at the `Collect` function:

```solidity
function Collect(uint _am)
public
payable
{
    var acc = Acc[msg.sender];
    if( acc.balance>=MinSum && acc.balance>=_am && now>acc.unlockTime)
    {
        if(msg.sender.call.value(_am)())
        {
            acc.balance-=_am;
            LogFile.AddMessage(msg.sender,_am,"Collect");
        }
    }
}
```

The vulnerability exists because:
1. The contract performs an external call with `msg.sender.call.value(_am)()` which can trigger code execution on the recipient address
2. The contract then updates the user's balance AFTER this call with `acc.balance-=_am;`
3. If the recipient is a malicious contract, its fallback fun

## Analyze the smart contract that need intervention with the help of Claude 3.7 Sonet LLM model

In [59]:
client = anthropic.Anthropic(
    # defaults to os.environ.get("ANTHROPIC_API_KEY")
    # api_key="",
)

In [61]:
system_prompt_template = "You are a blockchain security expert specializing in smart contract vulnerability analysis. Your task is to analyze Solidity smart contracts for {vulnerability} vulnerabilities."

prompt_part_1 = """
Analyze the smart contract for {vulnerability} vulnerabilities and identify which line or lines make the smart contract vulnerable to {vulnerability} attacks.  

Here's the scenario:

Contract code:
```
{contract_code}
```

"""
prompt_part_2 = """
Vulnerability detection tool #{tool_number} identified {this_line_or_these_lines} as vulnerable:
{line_or_lines}

"""

prompt_part_3 = """
Think before you give the final answer. First, think about what a {vulnerability} vulnerability is and what are the possible ways the {vulnerability} vulnerability can occur in a smart contract. Then, examine the smart contract code and the results of the vulnerability detection tools to guide your analysis of how {vulnerability} vulnerability can occur in the given smart contract, and identify the scenarios in which {vulnerability} vulnerability occurs. Then evaluate scenarios to verify they are correct. Then output only the line or lines that make the smart contract vulnerable to {vulnerability} attacks. 

Provide your response in exactly this format:
VULNERABLE_LINES: [List all the lines that create the {vulnerability} vulnerability a possibility in the given smart contract]
"""

## Helper functions

In [84]:
def prepare_prompt_part_2(detected_lines_list, source_code_lines, prompt, tool_number):
    lines_for_prompt_list = None
    if detected_lines_list is not None and len(detected_lines_list) > 0:
        lines_for_prompt = ""
        lines_for_prompt_list = []
        this_line_or_these_lines = ""
        detected_lines_list.sort()
        for i in range(len(detected_lines_list)):
            line = detected_lines_list[i]
            if len(detected_lines_list) == 1:
                lines_for_prompt += "```" + source_code_lines[line - 1].strip() + "```"
                lines_for_prompt_list.append(source_code_lines[line - 1].strip())
                this_line_or_these_lines = "this line"
            else:
                this_line_or_these_lines = "these lines"
                if i == (len(detected_lines_list) - 1):
                    lines_for_prompt += "```" + source_code_lines[line - 1].strip() + "```"
                    lines_for_prompt_list.append(source_code_lines[line - 1].strip())
                else:
                    lines_for_prompt += "```" + source_code_lines[line - 1].strip() + "```,\n"
                    lines_for_prompt_list.append(source_code_lines[line - 1].strip())
        prompt += prompt_part_2.format(tool_number=tool_number, this_line_or_these_lines=this_line_or_these_lines, line_or_lines=lines_for_prompt)
    return prompt, lines_for_prompt_list

def evaluate_set_string_to_list(set_string):
    if set_string != None:
        return list(eval(set_string))
    else:
        return None

def call_anthropic_api(prompt, system_prompt):
    response = client.messages.create(
    model="claude-3-7-sonnet-20250219",
    max_tokens=8000,
    system=system_prompt,
    thinking={
        "type": "enabled",
        "budget_tokens": 4000
    },
    messages=[{
        "role": "user",
        "content": prompt
    }])
    return response.content

def change_vulnerability_name_for_file(vulnerability):
    if vulnerability == "Arithmetic/Integer Overflow and Underflow":
        return "Integer_Overflow_and_Underflow"
    elif vulnerability == "Nested Call":
        return "Nested_Call"
    elif vulnerability == "Timestamp Dependency":
        return "Timestamp_Dependency"
    elif vulnerability == "Transaction Order Dependency":
        return "Transaction_Order_Dependency"
    elif vulnerability == "Unchecked Call":
        return "Unchecked_Call"
    elif vulnerability == "Unprotected Suicide":
        return "Unprotected_Suicide"
    elif vulnerability == "Frozen Ether":
        return "Frozen_Ether"
    else:
        return vulnerability

def ask_from_claude(df, original_df, vulnerability, dataset_name, call_api=False):
    pre_claude_results = []
    df = df[df['need_intervention'] == True]
    for index, row in df.iterrows():
        contract_address = row['contract_address']
        file_path = row['file_path']
        original_row = original_df[(original_df['contract_address'] == contract_address) & (original_df['file_path'] == file_path)]
        # check if there is only one row returned
        if len(original_row) == 0 or len(original_row) > 1:
            print(f"An unexpected error occurred! The length of original_row is either 0 or greater than 1. contract address: {contract_address}, file_path: {file_path}")
            continue
        # get the source code for the smart contract
        source_code = original_row.iloc[0]['source_code']

        # get the vulnerable lines identified by each tool
        soldetector_lines = evaluate_set_string_to_list(row['soldetector'])
        slither_lines = evaluate_set_string_to_list(row['slither'])
        oyente_lines = evaluate_set_string_to_list(row['oyente'])
        smartcheck_lines = evaluate_set_string_to_list(row['smartcheck'])

        ############### Create the prompt #####################

        prompt = prompt_part_1.format(vulnerability=vulnerability, contract_code=source_code)

        source_code_lines = source_code.splitlines()
        
        # Soldetector
        prompt, vuln_lines_soldetector = prepare_prompt_part_2(soldetector_lines, source_code_lines, prompt,  "1")

        # Slither
        prompt, vuln_lines_slither = prepare_prompt_part_2(slither_lines, source_code_lines, prompt, "2")

        # Oyente
        prompt, vuln_lines_oyente = prepare_prompt_part_2(oyente_lines, source_code_lines, prompt, "3")

        # Smartcheck
        prompt, vuln_lines_smartcheck = prepare_prompt_part_2(smartcheck_lines, source_code_lines, prompt, "4")

        prompt += prompt_part_3.format(vulnerability=vulnerability)

        ############### Create the system prompt #####################

        system_prompt = system_prompt_template.format(vulnerability=vulnerability)

        # print("prompt:\n---------------------------------------------------------------------------------------------------------------------")
        # print(prompt)
        # print("\n\nsystem_prompt:\n----------------------------------------------------------------------------------------------------------")
        # print(system_prompt)
        # print("\n\n========================================================================================================================\n\n")

        info_dict = {"file_path": file_path, "contract_address": contract_address, "soldetector": vuln_lines_soldetector, "slither": vuln_lines_slither, "oyente": vuln_lines_oyente, "smartcheck": vuln_lines_smartcheck, "prompt": prompt, "system_prompt": system_prompt}
        pre_claude_results.append(info_dict)

    post_claude_results = []

    curr_api_req_index = 0
    if call_api:
        for entry in pre_claude_results:
            curr_api_req_index += 1
            try:
                response = call_anthropic_api(entry['prompt'], entry['system_prompt'])
                print(f"{curr_api_req_index}/{len(pre_claude_results)} requests done")
            except:
                print(f"Error while getting response for file_path: {file_path}, contract_address: {contract_address} from Anthropic API")
                continue
            new_entry = dict(entry)
            new_entry['response'] = response
            post_claude_results.append(new_entry)
            time.sleep(20) # wait 10 seconds

    vulnerability_name = change_vulnerability_name_for_file(vulnerability)
            
    with open(f"post_claude_results_{dataset_name}_{vulnerability_name}.pkl", "wb") as file:
        pickle.dump(post_claude_results, file)
    with open(f"pre_claude_results_{dataset_name}_{vulnerability_name}.pkl", "wb") as file:
        pickle.dump(pre_claude_results, file)
    return pre_claude_results, post_claude_results        

In [74]:
# pre_claude_results, post_claude_results = ask_from_claude(result_dict_test['DC'], test, "DelegateCall", "test", call_api=True)

In [68]:
# print(post_claude_results[0]['response'][0].thinking)

### Train dataset

Number of contracts needing intervention:  267

Number of contracts needing intervention by vuln. type:  {'IOU': 30, 'TOD': 1, 'RE': 56, 'UcC': 0, 'UpS': 64, 'TD': 5, 'TO': 0, 'DC': 9, 'NC': 100, 'FE': 2}

In [86]:
dataset_name = "train"

# DC - DelegateCall 
_, _ = ask_from_claude(result_dict_train['DC'], train, "DelegateCall", dataset_name, call_api=True)

# IOU (Arithmetic/Integer Overflow and Underflow)
_, _ = ask_from_claude(result_dict_train['IOU'], train, "Arithmetic/Integer Overflow and Underflow", dataset_name, call_api=True)

# NC (Nested Call)
_, _ = ask_from_claude(result_dict_train['NC'], train, "Nested Call", dataset_name, call_api=True)

# RE (Reentrancy)
_, _ = ask_from_claude(result_dict_train['RE'], train, "Reentrancy", dataset_name, call_api=True)

# TD (Timestamp Dependency)
_, _ = ask_from_claude(result_dict_train['TD'], train, "Timestamp Dependency", dataset_name, call_api=True)

# TO (TxOrigin)
# _, _ = ask_from_claude(result_dict_train['TO'], train, "TxOrigin", dataset_name, call_api=True)

# TOD (Transaction Order Dependency)
_, _ = ask_from_claude(result_dict_train['TOD'], train, "Transaction Order Dependency", dataset_name, call_api=True)

# UcC (Unchecked Call)
# _, _ = ask_from_claude(result_dict_train['UcC'], train, "Unchecked Call", dataset_name, call_api=True)

# UpS (Unprotected Suicide)
_, _ = ask_from_claude(result_dict_train['UpS'], train, "Unprotected Suicide", dataset_name, call_api=True)

# FE (Frozen Ether)
_, _ = ask_from_claude(result_dict_train['FE'], train, "Frozen Ether", dataset_name, call_api=True)

1/9 requests done
2/9 requests done
3/9 requests done
4/9 requests done
5/9 requests done
6/9 requests done
7/9 requests done
8/9 requests done
9/9 requests done
1/30 requests done
2/30 requests done
3/30 requests done
4/30 requests done
5/30 requests done
6/30 requests done
7/30 requests done
8/30 requests done
9/30 requests done
10/30 requests done
11/30 requests done
12/30 requests done
13/30 requests done
14/30 requests done
15/30 requests done
16/30 requests done
17/30 requests done
18/30 requests done
19/30 requests done
20/30 requests done
21/30 requests done
22/30 requests done
23/30 requests done
24/30 requests done
25/30 requests done
26/30 requests done
27/30 requests done
28/30 requests done
29/30 requests done
30/30 requests done
1/100 requests done
2/100 requests done
3/100 requests done
4/100 requests done
5/100 requests done
6/100 requests done
7/100 requests done
8/100 requests done
9/100 requests done
10/100 requests done
11/100 requests done
12/100 requests done
13/1

### Test dataset

Number of contracts needing intervention:  16

Number of contracts needing intervention by vuln. type:  {'IOU': 2, 'TOD': 0, 'RE': 2, 'UcC': 0, 'UpS': 3, 'TD': 1, 'DC': 1, 'NC': 7}

In [80]:
dataset_name = "test"

# DC - DelegateCall
# _, _ = ask_from_claude(result_dict_test['DC'], test, "DelegateCall", dataset_name, call_api=True) -- Done

# IOU (Arithmetic/Integer Overflow and Underflow)
_, _ = ask_from_claude(result_dict_test['IOU'], test, "Arithmetic/Integer Overflow and Underflow", dataset_name, call_api=True)

# NC (Nested Call)
_, _ = ask_from_claude(result_dict_test['NC'], test, "Nested Call", dataset_name, call_api=True)

# RE (Reentrancy)
_, _ = ask_from_claude(result_dict_test['RE'], test, "Reentrancy", dataset_name, call_api=True)

# TD (Timestamp Dependency)
_, _ = ask_from_claude(result_dict_test['TD'], test, "Timestamp Dependency", dataset_name, call_api=True)

# TO (TxOrigin)
# _, _ = ask_from_claude(result_dict_test['TO'], test, "TxOrigin", dataset_name, call_api=True)

# TOD (Transaction Order Dependency)
# _, _ = ask_from_claude(result_dict_test['TOD'], test, "Transaction Order Dependency", dataset_name, call_api=True)

# UcC (Unchecked Call)
# _, _ = ask_from_claude(result_dict_test['UcC'], test, "Unchecked Call", dataset_name, call_api=True)

# UpS (Unprotected Suicide)
_, _ = ask_from_claude(result_dict_test['UpS'], test, "Unprotected Suicide", dataset_name, call_api=True)

# FE (Frozen Ether)
# _, _ = ask_from_claude(result_dict_test['FE'], test, "Frozen Ether", dataset_name, call_api=True)

1/2 requests done
2/2 requests done
1/7 requests done
2/7 requests done
3/7 requests done
4/7 requests done
5/7 requests done
6/7 requests done
7/7 requests done
1/2 requests done
2/2 requests done
1/1 requests done
1/3 requests done
2/3 requests done
3/3 requests done


### Validation dataset

Number of contracts needing intervention:  18

Number of contracts needing intervention by vuln. type:  {'IOU': 0, 'TOD': 0, 'RE': 1, 'UcC': 0, 'UpS': 5, 'TD': 0, 'TO': 0, 'DC': 1, 'NC': 11}

In [82]:
dataset_name = "validation"

# DC - DelegateCall
_, _ = ask_from_claude(result_dict_validation['DC'], validation, "DelegateCall", dataset_name, call_api=True)

# IOU (Arithmetic/Integer Overflow and Underflow)
# _, _ = ask_from_claude(result_dict_validation['IOU'], validation, "Arithmetic/Integer Overflow and Underflow", dataset_name, call_api=True)

# NC (Nested Call)
_, _ = ask_from_claude(result_dict_validation['NC'], validation, "Nested Call", dataset_name, call_api=True)

# RE (Reentrancy)
_, _ = ask_from_claude(result_dict_validation['RE'], validation, "Reentrancy", dataset_name, call_api=True)

# TD (Timestamp Dependency)
# _, _ = ask_from_claude(result_dict_validation['TD'], validation, "Timestamp Dependency", dataset_name, call_api=True)

# TO (TxOrigin)
# _, _ = ask_from_claude(result_dict_validation['TO'], validation, "TxOrigin", dataset_name, call_api=True)

# TOD (Transaction Order Dependency)
# _, _ = ask_from_claude(result_dict_validation['TOD'], validation, "Transaction Order Dependency", dataset_name, call_api=True)

# UcC (Unchecked Call)
# _, _ = ask_from_claude(result_dict_validation['UcC'], validation, "Unchecked Call", dataset_name, call_api=True)

# UpS (Unprotected Suicide)
_, _ = ask_from_claude(result_dict_validation['UpS'], validation, "Unprotected Suicide", dataset_name, call_api=True)

# FE (Frozen Ether)
# _, _ = ask_from_claude(result_dict_validation['FE'], validation, "Frozen Ether", dataset_name, call_api=True)

1/1 requests done
1/11 requests done
2/11 requests done
3/11 requests done
4/11 requests done
5/11 requests done
6/11 requests done
7/11 requests done
8/11 requests done
9/11 requests done
10/11 requests done
11/11 requests done
1/1 requests done
1/5 requests done
2/5 requests done
3/5 requests done
4/5 requests done
5/5 requests done


## Extract the vulnerable lines from LLM output and identify which line numbers they belong to in the source code

In [48]:
import pandas as pd
import pickle
import os
import json
import random
import numpy as np
import anthropic
import warnings
import time
import fnmatch
import re

with warnings.catch_warnings():
    warnings.simplefilter("ignore")

In [50]:
cwd = os.getcwd()

train = pd.read_parquet(f'{cwd}\\Vulnerable-Verified-Smart-Contracts\\train.parquet')
test = pd.read_parquet(f'{cwd}\\Vulnerable-Verified-Smart-Contracts\\test.parquet')
validation = pd.read_parquet(f'{cwd}\\Vulnerable-Verified-Smart-Contracts\\validation.parquet')

In [52]:
def add_new_columns(dataset):
    vulns = ['IOU', 'TOD', 'RE', 'UcC', 'UpS', 'TD', 'TO', 'DC', 'NC', 'FE']
    tools = ['soldetector', 'slither', 'oyente', 'smartcheck']

    # Initialize new columns
    for vuln in vulns:
        dataset[f'{vuln}'] = "Undetected"
        for tool in tools:
            dataset[f'{vuln}_{tool}'] = "Undetected"
            
    for index, row in dataset.iterrows():
        overlapping = row['overlapping']
    
        # Parse JSON string to Python dictionary
        data = json.loads(overlapping)
    
        vulns = data.keys()
        for vuln in vulns:
            info_dict_list = data[vuln][0]
            dataset.at[index, f'{vuln}'] = "Detected"
            for info_dict in info_dict_list:
                # print(info_dict)
                try:
                    lines = str(info_dict['lines'])
                except KeyError as e:
                    lines = "[" + str(info_dict['line']) + "]"
                tool = info_dict['tool']
                dataset.at[index, f'{vuln}_{tool}'] = lines

def check_for_improper_detections(dataset):
    vulns = ['IOU', 'TOD', 'RE', 'UcC', 'UpS', 'TD', 'TO', 'DC', 'NC', 'FE']
    tools = ['soldetector', 'slither', 'oyente', 'smartcheck']
    result = dict()
    
    for vuln in vulns:
        vuln_df = dataset[dataset[vuln] == "Detected"]

        if len(vuln_df) > 0:
            # print("Vulnerability: ", vuln, "Number of samples: ", len(vuln_df))
            df = pd.DataFrame(columns=["file_path", "contract_address", "soldetector", "slither", "oyente", "smartcheck", "need_intervention"])
            # Set all columns except 'need_intervention' to 'object' type
            df[df.columns.difference(['need_intervention'])] = df[df.columns.difference(['need_intervention'])].astype('object')
            
            row_index = 0
            for index, row in vuln_df.iterrows():
                df.loc[row_index] = [row['file_path'], row['contract_address'], None, None, None, None, False]
                vul_lines_set_for_all_tool = []
                need_intervention = False
                for tool in tools:
                    vul_lines_set_for_tool = set()
                    if row[f'{vuln}_{tool}'] != "Undetected":
                        line_list = eval(row[f'{vuln}_{tool}'])
                        vul_lines_set_for_tool.update(line_list)
                        vul_lines_set_for_all_tool.append(vul_lines_set_for_tool)
                        if len(vul_lines_set_for_tool) > 0:
                            df.loc[row_index, tool] = str(vul_lines_set_for_tool)
                for i in range(len(vul_lines_set_for_all_tool)):
                    for j in range(len(vul_lines_set_for_all_tool)):
                        if i == j:
                            continue
                        if vul_lines_set_for_all_tool[i] != vul_lines_set_for_all_tool[j]:
                            need_intervention = True
                df.loc[row_index, "need_intervention"] = need_intervention
                row_index += 1
            result[vuln] = df
    return result

def count_need_intervention(result_dict):
    # print("Vulberability types: ", list(result_dict.keys()))
    # print("Number of vulnerability types: ", len(result_dict.keys()))
    count_dict = dict()
    count_sum = 0
    for key in result_dict.keys():
        df = result_dict[key]
        true_count = (df['need_intervention'] == True).sum()
        count_sum += true_count
        count_dict[key] = true_count
    return count_sum, count_dict

In [54]:
# Add new columns to the dataframes for further processing
add_new_columns(train)
add_new_columns(test)
add_new_columns(validation)

In [56]:
result_dict_train = check_for_improper_detections(train)
result_dict_test = check_for_improper_detections(test)
result_dict_validation = check_for_improper_detections(validation)

count_train, count_dict_train = count_need_intervention(result_dict_train)
count_test, count_dict_test = count_need_intervention(result_dict_test)
count_validation, count_dict_validation = count_need_intervention(result_dict_validation)

print("Train dataset\n===============================\n")
print("Number of contracts needing intervention: ", count_train)
print("Number of contracts needing intervention by vuln. type: ", count_dict_train)

print("\n\nTest dataset\n===============================\n")
print("Number of contracts needing intervention: ", count_test)
print("Number of contracts needing intervention by vuln. type: ", count_dict_test)

print("\n\nValidation dataset\n===============================")
print("Number of contracts needing intervention: ", count_validation)
print("Number of contracts needing intervention by vuln. type: ", count_dict_validation)

Train dataset

Number of contracts needing intervention:  267
Number of contracts needing intervention by vuln. type:  {'IOU': 30, 'TOD': 1, 'RE': 56, 'UcC': 0, 'UpS': 64, 'TD': 5, 'TO': 0, 'DC': 9, 'NC': 100, 'FE': 2}


Test dataset

Number of contracts needing intervention:  16
Number of contracts needing intervention by vuln. type:  {'IOU': 2, 'TOD': 0, 'RE': 2, 'UcC': 0, 'UpS': 3, 'TD': 1, 'DC': 1, 'NC': 7}


Validation dataset
Number of contracts needing intervention:  18
Number of contracts needing intervention by vuln. type:  {'IOU': 0, 'TOD': 0, 'RE': 1, 'UcC': 0, 'UpS': 5, 'TD': 0, 'TO': 0, 'DC': 1, 'NC': 11}


### Load saved vulnerability identification results

In [59]:
vulns_mapping = {'IOU': "Integer_Overflow_and_Underflow", 'TOD': "Transaction_Order_Dependency", 'RE': "Reentrancy", 'UcC': "Unchecked_Call", 'UpS': "Unprotected_Suicide", 'TD': "Timestamp_Dependency", 'TO': "TxOrigin", 'DC': "DelegateCall", 'NC': "Nested_Call", 'FE': "Frozen_Ether"}
reverse_vulns_mapping = {v: k for k, v in vulns_mapping.items()}
print(reverse_vulns_mapping)

{'Integer_Overflow_and_Underflow': 'IOU', 'Transaction_Order_Dependency': 'TOD', 'Reentrancy': 'RE', 'Unchecked_Call': 'UcC', 'Unprotected_Suicide': 'UpS', 'Timestamp_Dependency': 'TD', 'TxOrigin': 'TO', 'DelegateCall': 'DC', 'Nested_Call': 'NC', 'Frozen_Ether': 'FE'}


In [61]:
all_pkl_files = [f for f in os.listdir(cwd) if fnmatch.fnmatch(f, "*.pkl")]
results_pkl_files = []
for file in all_pkl_files:
    if "post_claude_results_" in file:
        results_pkl_files.append(file)
train_results_pkl_files = dict()
test_results_pkl_files = dict()
validation_results_pkl_files = dict()

vulns_namings = list(reverse_vulns_mapping.keys())
def check_which_vuln_and_get_key(file_path):
    for vuln_name in vulns_namings:
        if "_" + vuln_name + ".pkl" in file_path:
            return reverse_vulns_mapping[vuln_name]
    
for file in results_pkl_files:
    key = check_which_vuln_and_get_key(file)
    if "_train_" in file:
        train_results_pkl_files[key] = file
    elif "_test_" in file:
        test_results_pkl_files[key] = file
    elif "_validation_" in file:
        validation_results_pkl_files[key] = file

print(train_results_pkl_files, len(train_results_pkl_files))
print(test_results_pkl_files, len(test_results_pkl_files))
print(validation_results_pkl_files, len(validation_results_pkl_files))

{'DC': 'post_claude_results_train_DelegateCall.pkl', 'FE': 'post_claude_results_train_Frozen_Ether.pkl', 'IOU': 'post_claude_results_train_Integer_Overflow_and_Underflow.pkl', 'NC': 'post_claude_results_train_Nested_Call.pkl', 'RE': 'post_claude_results_train_Reentrancy.pkl', 'TD': 'post_claude_results_train_Timestamp_Dependency.pkl', 'TOD': 'post_claude_results_train_Transaction_Order_Dependency.pkl', 'UpS': 'post_claude_results_train_Unprotected_Suicide.pkl'} 8
{'DC': 'post_claude_results_test_DelegateCall.pkl', 'IOU': 'post_claude_results_test_Integer_Overflow_and_Underflow.pkl', 'NC': 'post_claude_results_test_Nested_Call.pkl', 'RE': 'post_claude_results_test_Reentrancy.pkl', 'TD': 'post_claude_results_test_Timestamp_Dependency.pkl', 'UpS': 'post_claude_results_test_Unprotected_Suicide.pkl'} 6
{'DC': 'post_claude_results_validation_DelegateCall.pkl', 'NC': 'post_claude_results_validation_Nested_Call.pkl', 'RE': 'post_claude_results_validation_Reentrancy.pkl', 'UpS': 'post_claude_re

In [63]:
def check_result_output_formatting(file_path):
    not_according_to_format = 0
    not_according_to_format_loosed = 0
    with open(file_path, "rb") as file:
        data = pickle.load(file)  # Load the pickled data
    
        for item in data:
            response = item['response']
            text_portion = response[-1].text
            if "VULNERABLE_LINES: [" not in text_portion:
                not_according_to_format += 1
                # print(item['file_path'], item['contract_address'])
                # print(text_portion)
                # print("===========================================================================================\n\n")
            if "VULNERABLE_LINES: " not in text_portion:
                not_according_to_format_loosed += 1
        print("Tight formatting: ", file_path, not_according_to_format)
        print("Loose formatting: ", file_path, not_according_to_format_loosed)
        print()

def check_result_output_formatting_in_all():
    for file_path in results_pkl_files:
        check_result_output_formatting(file_path)

check_result_output_formatting_in_all()

Tight formatting:  post_claude_results_test_DelegateCall.pkl 0
Loose formatting:  post_claude_results_test_DelegateCall.pkl 0

Tight formatting:  post_claude_results_test_Integer_Overflow_and_Underflow.pkl 0
Loose formatting:  post_claude_results_test_Integer_Overflow_and_Underflow.pkl 0

Tight formatting:  post_claude_results_test_Nested_Call.pkl 0
Loose formatting:  post_claude_results_test_Nested_Call.pkl 0

Tight formatting:  post_claude_results_test_Reentrancy.pkl 0
Loose formatting:  post_claude_results_test_Reentrancy.pkl 0

Tight formatting:  post_claude_results_test_Timestamp_Dependency.pkl 0
Loose formatting:  post_claude_results_test_Timestamp_Dependency.pkl 0

Tight formatting:  post_claude_results_test_Unprotected_Suicide.pkl 0
Loose formatting:  post_claude_results_test_Unprotected_Suicide.pkl 0

Tight formatting:  post_claude_results_train_DelegateCall.pkl 0
Loose formatting:  post_claude_results_train_DelegateCall.pkl 0

Tight formatting:  post_claude_results_train_Froz

In [65]:
def extract_vulnerable_lines(text):
    match = re.search(r"VULNERABLE_LINES:(.*)", text, re.DOTALL)
    if match:
        return match.group(1)  # Extract the content inside brackets
    return None

def check_results(file_path):
    not_according_to_format = 0
    not_according_to_format_loosed = 0
    with open(file_path, "rb") as file:
        data = pickle.load(file)  # Load the pickled data
    
        for item in data:
            response = item['response']
            text_portion = response[-1].text
            extracted_match = extract_vulnerable_lines(text_portion)
            print(extracted_match, item['file_path'], item['contract_address'])
            print()

def check_results_in_all():
    for file_path in results_pkl_files:
        check_results(file_path)

check_results_in_all()

 [607-608, 624-625, 680-681] Pair.sol 0x0b3ded1b54e89f8899482cfccf515d525835bc2b

 [return denominator * 100 * address(this).balance / targetAmount;, emit Transfer(address(this), targetAddress, denominator * msg.value / targetAmount * 100);] PiggyBank.sol 0x7a0ec191d8f44f3bc3cbb4d1105ced31c30ff492

 [uint256 newToVal = oldToVal + _value;]

While there is a check (`require(newToVal > oldToVal);`) that helps detect overflow, this approach isn't foolproof in all scenarios and could potentially be circumvented depending on implementation details. TokenTycoonIGO.sol 0xfb673f08fc82807b4d0e139e794e3b328d63551f

 [for(uint winList = 0; winList < winners.length; winList++){ , winners[winList].transfer(bonusETH.div(64));] CrystalReignShard.sol 0x5108e432d40d34a4d0ed88c12e83db0e571bfc75

 [for (uint256 i = 0; i < list.length; i++) {, address(list[i]).transfer(amount);] MultiEthSender.sol 0x11352d75f63804d5f8fdb88bddf9efe9b52d6723

 [_contributors[i].transfer(_balances[i]);, for (i; i < _contribut

### Create a list to hold the smart contracts that need to be omitted going forward

In [68]:
contracts_to_omit_ongoing = []

### Separate the results: []

In [71]:
def extract_vulnerable_lines(text):
    match = re.search(r"VULNERABLE_LINES:(.*)", text, re.DOTALL)
    if match:
        return match.group(1)  # Extract the content inside brackets
    return None

def check_results(file_path):
    not_according_to_format = 0
    not_according_to_format_loosed = 0
    with open(file_path, "rb") as file:
        data = pickle.load(file)  # Load the pickled data
    
        for item in data:
            response = item['response']
            text_portion = response[-1].text
            extracted_match = extract_vulnerable_lines(text_portion).strip()
            if extracted_match == "[]" or extracted_match == None:
                print(item['file_path'], item['contract_address'], file_path)
                print(text_portion)
                print("==============================================================================================================\n\n")

def check_results_in_all():
    for file_path in results_pkl_files:
        check_results(file_path)

check_results_in_all()

EthealSplit.sol 0x99d804f479df333ed4d2287af2d4da3eda1b3cd1 post_claude_results_train_Integer_Overflow_and_Underflow.pkl
I need to analyze the provided contract for arithmetic overflow/underflow vulnerabilities in Solidity.

First, let me understand what these vulnerabilities are:
- Arithmetic overflow occurs when a number exceeds its maximum value and wraps around
- Arithmetic underflow occurs when a number goes below its minimum value and wraps around
- In Solidity versions before 0.8.0, these aren't checked automatically

Looking at the EthealSplit contract (on Solidity 0.4.17):

1. The division operation `msg.value / _to.length` cannot cause an overflow/underflow itself, but could cause a division by zero if `_to.length` is 0

2. The loop counter `i` increases with each iteration, but for it to overflow, the array length would need to be greater than 2^256-1, which is practically impossible

3. No other arithmetic operations appear in the contract

While the contract has other poten

**We will drop these two vulnerabilities from the dataset as it is recognized as false positive and human verification deemed the the logic also point to these being false positive**
- dataset - train
- vuln - IOU

1) EthealSplit.sol 0x99d804f479df333ed4d2287af2d4da3eda1b3cd1
2) Send.sol 0xf9e7083fd1ca37db176046ba17cc0fede7fa855d

In [74]:
df = result_dict_train['IOU']
filtered_df_1 = df[(df['file_path'] ==  "EthealSplit.sol") & (df['contract_address'] == "0x99d804f479df333ed4d2287af2d4da3eda1b3cd1")]
filtered_df_2 = df[(df['file_path'] ==  "Send.sol") & (df['contract_address'] == "0xf9e7083fd1ca37db176046ba17cc0fede7fa855d")]
# print(filtered_df_1)
# print(filtered_df_2)

dropped_df = df.drop(filtered_df_1.index).drop(filtered_df_2.index)

filtered_df_1 = dropped_df[(dropped_df['file_path'] ==  "EthealSplit.sol") & (dropped_df['contract_address'] == "0x99d804f479df333ed4d2287af2d4da3eda1b3cd1")]
filtered_df_2 = dropped_df[(dropped_df['file_path'] ==  "Send.sol") & (dropped_df['contract_address'] == "0xf9e7083fd1ca37db176046ba17cc0fede7fa855d")]

# print(filtered_df_1)
# print(filtered_df_2)

result_dict_train['IOU'] = dropped_df

In [76]:
contracts_to_omit_ongoing.append({'file_path': "EthealSplit.sol", "contract_address": "0x99d804f479df333ed4d2287af2d4da3eda1b3cd1", "vuln": "IOU"})
contracts_to_omit_ongoing.append({'file_path': "Send.sol", "contract_address": "0xf9e7083fd1ca37db176046ba17cc0fede7fa855d", "vuln": "IOU"})

### Separate the results: [line numbers]

In [80]:
def extract_vulnerable_lines_that_give_a_list_of_numbers(text):
    match = re.search(r"VULNERABLE_LINES:\s*(\[?[0-9])", text, re.DOTALL)
    if match:
        return match.group(1)  # Extract the content inside brackets
    return None

vulns_namings = list(reverse_vulns_mapping.keys())
def check_which_vuln_and_get_key(file_path):
    for vuln_name in vulns_namings:
        if "_" + vuln_name + ".pkl" in file_path:
            return reverse_vulns_mapping[vuln_name]

def extract_vulnerable_lines(text):
    match = re.search(r"VULNERABLE_LINES:\s*(.*)", text, re.DOTALL)
    if match:
        return match.group(1)  # Extract the content inside brackets
    return None

items_with_prompts = []
def check_results(file_path):
    not_according_to_format = 0
    not_according_to_format_loosed = 0
    with open(file_path, "rb") as file:
        data = pickle.load(file)  # Load the pickled data
    
        for item in data:
            response = item['response']
            text_portion = response[-1].text
            extracted_match = extract_vulnerable_lines_that_give_a_list_of_numbers(text_portion)
            if extracted_match != None:
                correct_match = extract_vulnerable_lines(text_portion)
                vuln_key = check_which_vuln_and_get_key(file_path)
                contracts_to_omit_ongoing.append({'file_path': item['file_path'], "contract_address": item['contract_address'], "vuln": vuln_key})
                print(correct_match, item['file_path'], item['contract_address'], file_path)
                print(text_portion)
                items_with_prompts.append(item)
                print("==============================================================================================================\n\n")

def check_results_in_all():
    for file_path in results_pkl_files:
        check_results(file_path)

check_results_in_all()

[607-608, 624-625, 680-681] Pair.sol 0x0b3ded1b54e89f8899482cfccf515d525835bc2b post_claude_results_test_DelegateCall.pkl
Looking at this smart contract for delegatecall vulnerabilities, I need to understand what makes delegatecall dangerous and where it's used in this code.

When a contract uses delegatecall, it executes another contract's code within its own context, meaning:
1. The external code can access and modify the calling contract's storage
2. The external code inherits the same permissions and state of the calling contract
3. Storage layout collisions can occur if the contracts don't align their variables

In this contract, I can see three instances of delegatecall in the Pair contract:

1. In the `short` function at line 607-608:
```solidity
(bool success, bytes memory result) = swapper.delegatecall(
    abi.encodeWithSignature("swap(address,address,address,uint256,uint256)", swapper, tokenAsset, tokenCollateral, amountAsset, minAmountCollateral));
```

2. In the `unwind` f

In [82]:
print(len(items_with_prompts))

14


In [118]:
index__ = 13

print(items_with_prompts[index__]['prompt'])
print("\n\n\n\n\n\n\n\n", items_with_prompts[index__]['file_path'])
print(items_with_prompts[index__]['contract_address'])


Analyze the smart contract for Nested Call vulnerabilities and identify which line or lines make the smart contract vulnerable to Nested Call attacks.  

Here's the scenario:

Contract code:
```
pragma solidity >=0.4.23 <0.6.0;



contract SkyWay {
 
    struct User {
        uint id;
        address referrer;
        uint partnersCount;
        
        mapping(uint8 => bool) activeX3Levels;
        mapping(uint8 => bool) activeX6Levels;
        
        mapping(uint8 => X3) x3Matrix;
        mapping(uint8 => X6) x6Matrix;
      
    }
    
    struct X3 {
        address currentReferrer;
        address[] referrals;
        bool blocked;
        uint reinvestCount;
    }
    
    struct X6 {
        address currentReferrer;
        address[] firstLevelReferrals;
        address[] secondLevelReferrals;
        bool blocked;
        uint reinvestCount;

        address closedPart;
    }
    
    
      
    uint8 public constant LAST_LEVEL = 14;
    
    mapping(address => User) publi

#### Identify the vulnerable lines with the help of LLM output and human intervention

In [42]:
# result_dict_test['DC']
# source = test[test['file_path'] == "Pair.sol"].iloc[0]['source_code']
# print(source)

### Identify other vulnerable lines using automation methods

#### Where extracted match is covered by a set of brackets

In [46]:
def find_line_numbers(source_text, lines_to_find):
    """
    Find line numbers in source text where each line appears.
    Handles whitespace variations and multi-line spans.
    
    Args:
        source_text (str): The source text to search in
        lines_to_find (list): List of strings to find in the source
        
    Returns:
        dict: Dictionary mapping each search line to its line number(s) in source
    """
    # Normalize the source text by splitting into lines and stripping whitespace
    source_lines = [line.strip() for line in source_text.split('\n')]
    
    # Create a single string of normalized source with line numbers for debugging
    normalized_source = '\n'.join(f"{i+1}: {line}" for i, line in enumerate(source_lines))
    
    results = {}
    
    for line_to_find in lines_to_find:
        # Normalize the line to find by stripping whitespace
        normalized_line = line_to_find.strip()
        found_at = []
        
        # Check for exact matches first (single line)
        for i, source_line in enumerate(source_lines):
            if normalized_line == source_line:
                found_at.append(i+1)  # +1 because line numbers start at 1
        
        # If no exact matches, look for the line as part of multi-line content
        if not found_at:
            # Join the source text with spaces instead of newlines for a continuous search
            continuous_source = ' '.join(source_lines)
            continuous_line_map = []
            
            # Create mapping from position in continuous text to original line number
            current_pos = 0
            for i, line in enumerate(source_lines):
                line_length = len(line)
                if line:  # Only add non-empty lines
                    continuous_line_map.append((current_pos, current_pos + line_length, i+1))
                    current_pos += line_length + 1  # +1 for the space we added
            
            # Find all occurrences of the search string in the continuous text
            search_pos = 0
            while True:
                pos = continuous_source.find(normalized_line, search_pos)
                if pos == -1:
                    break
                
                # Find which line(s) this position corresponds to
                line_numbers = set()
                for start, end, line_num in continuous_line_map:
                    # Check if there's any overlap between search string and this line
                    if not (pos + len(normalized_line) <= start or pos >= end):
                        line_numbers.add(line_num)
                
                if line_numbers:
                    found_at.extend(sorted(line_numbers))
                
                search_pos = pos + 1  # Move search position forward
        
        results[line_to_find] = found_at
    
    return results

In [48]:
vulns_namings = list(reverse_vulns_mapping.keys())
def check_which_vuln_and_get_key(file_path):
    for vuln_name in vulns_namings:
        if "_" + vuln_name + ".pkl" in file_path:
            return reverse_vulns_mapping[vuln_name]
            
def extract_vulnerable_lines(text):
    match = re.search(r"VULNERABLE_LINES:(.*)", text, re.DOTALL)
    if match:
        return match.group(1)  # Extract the content inside brackets
    return None

def is_omitted(file_path, contract_address, pkl_file_path):
    for omitted in contracts_to_omit_ongoing:
        if omitted['file_path'] == file_path and omitted['contract_address'] == contract_address and omitted['vuln'] == check_which_vuln_and_get_key(pkl_file_path):
            return True
    return False

def seperate(text):
    if text[0] == '[':
        text = text[1:]
        if text[-1] == ']':
            text = text[:-1]
            comma_seperations = text.split(',')
            resultant_list = []
            for item in comma_seperations:
                item = item.strip()
                resultant_list.append(item)
            return resultant_list
        else:
            return None
    else:
        return None

def check_results(file_path):
    not_according_to_format = 0
    not_according_to_format_loosed = 0

    columns = [
    "file_path",
    "contract_address",
    "pkl_file_path",
    "extracted_match",
    "separation_output",
    "correct",
    "edited_separation_output"]
    
    # Create an empty DataFrame
    df = pd.DataFrame(columns=columns)

    with open(file_path, "rb") as file:
        data = pickle.load(file)  # Load the pickled data
    
        for item in data:
            if not is_omitted(item['file_path'], item['contract_address'], file_path):
                response = item['response']
                text_portion = response[-1].text
                extracted_match = extract_vulnerable_lines(text_portion).strip()
                if extracted_match[-1] == ']' and extracted_match[0] == '[':
                    # print(extracted_match, item['file_path'], item['contract_address'], file_path)
                    # print(text_portion)
                    # print("==============================================================================================================\n\n")

                    separation_output = seperate(extracted_match)
                    source_code = None
                    if "_train_" in file_path:
                        source_code = train[(train['file_path'] == item['file_path']) & (train['contract_address'] == item['contract_address'])].iloc[0]['source_code']
                    elif "_test_" in file_path:
                        source_code = test[(test['file_path'] == item['file_path']) & (test['contract_address'] == item['contract_address'])].iloc[0]['source_code']
                    elif "_validation_" in file_path:
                        source_code = validation[(validation['file_path'] == item['file_path']) & (validation['contract_address'] == item['contract_address'])].iloc[0]['source_code']
                    else:
                        print("Error while getting the source code", item['file_path'], item['contract_address'])
                    save_ready_seperation_output = ""
                    for line in separation_output:
                        save_ready_seperation_output += line + "\n"
                    new_row = [(item['file_path'], item['contract_address'], file_path, extracted_match, save_ready_seperation_output, 0, "")]
                    df.loc[len(df)] = new_row[0]
                    # if separation_output != None and source_code != None:
                    #     result = find_line_numbers(source_code, separation_output)
                    #     # Print results
                    #     for line, line_numbers in result.items():
                    #         if line_numbers:
                    #             print(f"Found: '{line}' found at line(s): {', '.join(map(str, line_numbers))}")
                    #         else:
                    #             print(f"Not Found: '{line}' not found")
                    # else:
                    #     print("Erorr: either seperation output is none or source code is none")
                    # print()
                else:
                    new_row = [(item['file_path'], item['contract_address'], file_path, extracted_match, "", 0, "")]
                    df.loc[len(df)] = new_row[0]
    return df

def check_results_in_all():
    all_results = []  # List to store DataFrames from each file

    for file_path in results_pkl_files:
        df = check_results(file_path)
        all_results.append(df)  # Collect DataFrames

    return pd.concat(all_results, ignore_index=True) if all_results else pd.DataFrame()  # Return combined DataFrame

df = check_results_in_all()

In [249]:
df.to_csv("identified_vuln_lines_by_llm.csv")

### Continue from hand cleaned LLM output

In [449]:
import pandas as pd
import pickle
import os
import json
import random
import numpy as np
import anthropic
import warnings
import time
import fnmatch
import re

with warnings.catch_warnings():
    warnings.simplefilter("ignore")

In [451]:
df = pd.read_csv("identified_vuln_lines_by_llm_modified.csv")

In [453]:
df

,Unnamed: 0,file_path,contract_address,pkl_file_path,extracted_match,separation_output,correct,edited_separation_output
0,0,PiggyBank.sol,0x7a0ec191d8f44f3bc3cbb4d1105ced31c30ff492,post_claude_results_test_Integer_Overflow_and_...,[return denominator * 100 * address(this).bala...,return denominator * 100 * address(this).balan...,1,NaN
1,1,TokenTycoonIGO.sol,0xfb673f08fc82807b4d0e139e794e3b328d63551f,post_claude_results_test_Integer_Overflow_and_...,[uint256 newToVal = oldToVal + _value;]\n\nWhi...,uint256 newToVal = oldToVal + _value;,1,NaN
2,2,CrystalReignShard.sol,0x5108e432d40d34a4d0ed88c12e83db0e571bfc75,post_claude_results_test_Nested_Call.pkl,[for(uint winList = 0; winList < winners.lengt...,for(uint winList = 0; winList < winners.length...,1,NaN
3,3,MultiEthSender.sol,0x11352d75f63804d5f8fdb88bddf9efe9b52d6723,post_claude_results_test_Nested_Call.pkl,"[for (uint256 i = 0; i < list.length; i++) {, ...",for (uint256 i = 0; i < list.length; i++) {\na...,1,NaN
4,4,eTrustmoney.sol,0xa5c62860475ce15c5e38a8443ca53f72113c0a5c,post_claude_results_test_Nested_Call.pkl,"[_contributors[i].transfer(_balances[i]);, for...",_contributors[i].transfer(_balances[i]);\nfor ...,1,NaN
...,...,...,...,...,...,...,...,...
280,280,Auctionify.sol,0x3a6984181f49c9ef64ada32373e736db0c9f1078,post_claude_results_validation_Unprotected_Sui...,[function cleanUpAfterYourself() public {],function cleanUpAfterYourself() public {\n,1,NaN
281,281,GetsBurned.sol,0x2f976a5382c0c6214b3d8fe6bdc5379edd35d9d4,post_claude_results_validation_Unprotected_Sui...,"[function BurnMe() public {, selfdestruct(addr...",function BurnMe() public {\nselfdestruct(addre...,1,NaN
282,282,WhoWillRuleWesterosAtTheEnd.sol,0xf2e88e0bfe61e5e41d9317e82c6938e67a913cc1,post_claude_results_validation_Unprotected_Sui...,[function clearBlockchain() external {],function clearBlockchain() external {\n,1,NaN
283,283,Promise.sol,0x23e6311dd0eb3556b300404f0cba576d504be223,post_claude_results_validation_Unprotected_Sui...,"[function selfDestruct() public{, selfdestruct...",function selfDestruct() public{\nselfdestruct(...,1,NaN


In [455]:
df['vuln'] = None
df['line_numbers'] = "Not_Found"
df = df.drop(columns=['edited_separation_output'])
df['line_numbers_complete'] = False
for index, row in df.iterrows():
    if row['correct'] == 0:
        continue
    separation_output = row['separation_output']
    a = separation_output
    separation_output = separation_output.splitlines()
    for i in range(len(separation_output)):
        separation_output[i] = separation_output[i].strip()
    file_path = row['pkl_file_path']
    item = row
    source_code = None
    if "_train_" in file_path:
        source_code = train[(train['file_path'] == item['file_path']) & (train['contract_address'] == item['contract_address'])].iloc[0]['source_code']
    elif "_test_" in file_path:
        source_code = test[(test['file_path'] == item['file_path']) & (test['contract_address'] == item['contract_address'])].iloc[0]['source_code']
    elif "_validation_" in file_path:
        source_code = validation[(validation['file_path'] == item['file_path']) & (validation['contract_address'] == item['contract_address'])].iloc[0]['source_code']
    else:
        print("Error while getting the source code", item['file_path'], item['contract_address'])

    df.loc[index, 'vuln'] = check_which_vuln_and_get_key(file_path)
    # save_ready_seperation_output = ""
    # for line in separation_output:
    #     save_ready_seperation_output += line + "\n"
    # new_row = [(item['file_path'], item['contract_address'], file_path, extracted_match, save_ready_seperation_output, 0, "")]
    # df.loc[len(df)] = new_row[0]
    line_numbers_set = set()
    is_some_not_found = False
    if separation_output != None and source_code != None:
        result = find_line_numbers(source_code, separation_output)
        # Print results
        for line, line_numbers in result.items():
            if line_numbers:
                line_numbers_set.update(line_numbers)
                print(f"Found: '{line}' found at line(s): {', '.join(map(str, line_numbers))}")
            else:
                is_some_not_found = True
                print(f"Not Found: '{line}' not found")
    else:
        print("Erorr: either seperation output is none or source code is none")
    df.loc[index, 'line_numbers'] = str(line_numbers_set)
    if item['file_path'] == "Amp.sol":
        print(line_numbers_set)
    if not is_some_not_found:
        df.loc[index, 'line_numbers_complete'] = True
    print()

Found: 'return denominator * 100 * address(this).balance / targetAmount;' found at line(s): 33
Found: 'emit Transfer(address(this), targetAddress, denominator * msg.value / targetAmount * 100);' found at line(s): 39

Found: 'uint256 newToVal = oldToVal + _value;' found at line(s): 159

Found: 'for(uint winList = 0; winList < winners.length; winList++){' found at line(s): 220
Found: 'winners[winList].transfer(bonusETH.div(64));' found at line(s): 221

Found: 'for (uint256 i = 0; i < list.length; i++) {' found at line(s): 79
Found: 'address(list[i]).transfer(amount);' found at line(s): 80

Found: '_contributors[i].transfer(_balances[i]);' found at line(s): 60
Found: 'for (i; i < _contributors.length; i++) {' found at line(s): 57

Found: '_to[i].transfer(_value[i]);' found at line(s): 35

Found: 'players[i].transfer(playerInfo[players[i]].betAmount);' found at line(s): 486
Found: 'players[i].transfer(winOdds.mul(playerInfo[players[i]].betAmount).div(100));' found at line(s): 511

Found: '

In [459]:
df

,Unnamed: 0,file_path,contract_address,pkl_file_path,extracted_match,separation_output,correct,vuln,line_numbers,line_numbers_complete
0,0,PiggyBank.sol,0x7a0ec191d8f44f3bc3cbb4d1105ced31c30ff492,post_claude_results_test_Integer_Overflow_and_...,[return denominator * 100 * address(this).bala...,return denominator * 100 * address(this).balan...,1,IOU,"{33, 39}",True
1,1,TokenTycoonIGO.sol,0xfb673f08fc82807b4d0e139e794e3b328d63551f,post_claude_results_test_Integer_Overflow_and_...,[uint256 newToVal = oldToVal + _value;]\n\nWhi...,uint256 newToVal = oldToVal + _value;,1,IOU,{159},True
2,2,CrystalReignShard.sol,0x5108e432d40d34a4d0ed88c12e83db0e571bfc75,post_claude_results_test_Nested_Call.pkl,[for(uint winList = 0; winList < winners.lengt...,for(uint winList = 0; winList < winners.length...,1,NC,"{220, 221}",True
3,3,MultiEthSender.sol,0x11352d75f63804d5f8fdb88bddf9efe9b52d6723,post_claude_results_test_Nested_Call.pkl,"[for (uint256 i = 0; i < list.length; i++) {, ...",for (uint256 i = 0; i < list.length; i++) {\na...,1,NC,"{80, 79}",True
4,4,eTrustmoney.sol,0xa5c62860475ce15c5e38a8443ca53f72113c0a5c,post_claude_results_test_Nested_Call.pkl,"[_contributors[i].transfer(_balances[i]);, for...",_contributors[i].transfer(_balances[i]);\nfor ...,1,NC,"{57, 60}",True
...,...,...,...,...,...,...,...,...,...,...
280,280,Auctionify.sol,0x3a6984181f49c9ef64ada32373e736db0c9f1078,post_claude_results_validation_Unprotected_Sui...,[function cleanUpAfterYourself() public {],function cleanUpAfterYourself() public {\n,1,UpS,{173},True
281,281,GetsBurned.sol,0x2f976a5382c0c6214b3d8fe6bdc5379edd35d9d4,post_claude_results_validation_Unprotected_Sui...,"[function BurnMe() public {, selfdestruct(addr...",function BurnMe() public {\nselfdestruct(addre...,1,UpS,{6},False
282,282,WhoWillRuleWesterosAtTheEnd.sol,0xf2e88e0bfe61e5e41d9317e82c6938e67a913cc1,post_claude_results_validation_Unprotected_Sui...,[function clearBlockchain() external {],function clearBlockchain() external {\n,1,UpS,{203},True
283,283,Promise.sol,0x23e6311dd0eb3556b300404f0cba576d504be223,post_claude_results_validation_Unprotected_Sui...,"[function selfDestruct() public{, selfdestruct...",function selfDestruct() public{\nselfdestruct(...,1,UpS,{91},False


In [461]:
(df['line_numbers_complete'] == False).sum()

26

### Merge cleaned LLM output with outputs from other vulnerability detectino tools for evaluation

In [464]:
df_save_names = []
def merge(curr_dict, curr_dict_name):
    print(curr_dict_name, "============================================================\n")
    for key in list(curr_dict.keys()):
        vuln_name = vulns_mapping[key]
        file_path = f"post_claude_results_{curr_dict_name}_{vuln_name}.pkl"
        filtered_df = df[df['pkl_file_path'] == file_path]
        print(key, len(filtered_df))
        vuln_dataset_df = curr_dict[key].copy()
        vuln_dataset_df['claude'] = None
        vuln_dataset_df['claude_complete'] = False
        vuln_dataset_df['pkl_file_path'] = None
        for index, row in filtered_df.iterrows():
            row_to_update = vuln_dataset_df[(vuln_dataset_df['file_path'] == row['file_path']) & (vuln_dataset_df['contract_address'] == row['contract_address'])]
            if len(row_to_update) > 1:
                print("There are more than one row to update. Something wrong!!!")
            update_index = row_to_update.index[0]
            vuln_dataset_df.loc[update_index, 'claude'] = row['line_numbers']
            vuln_dataset_df.loc[update_index, 'claude_complete'] = row['line_numbers_complete']
            vuln_dataset_df.loc[update_index, 'pkl_file_path'] = row['pkl_file_path']
        curr_dict[key] = vuln_dataset_df
        vuln_dataset_df.to_csv(f"{curr_dict_name}_{key}.csv")
        df_save_names.append(f"{curr_dict_name}_{key}.csv")
    print()

merge(result_dict_train, "train")
merge(result_dict_test, "test")
merge(result_dict_validation, "validation")

train ============================================================

IOU 25
TOD 1
RE 56
UcC 0
UpS 63
TD 5
TO 0
DC 8
NC 93
FE 2

test ============================================================

IOU 2
TOD 0
RE 2
UcC 0
UpS 3
TD 1
DC 0
NC 7

validation ============================================================

IOU 0
TOD 0
RE 1
UcC 0
UpS 5
TD 0
TO 0
DC 1
NC 10



In [468]:
check_df = result_dict_train['IOU']
check_df = check_df[check_df['need_intervention'] == True]
check_df

,file_path,contract_address,soldetector,slither,oyente,smartcheck,need_intervention,claude,claude_complete,pkl_file_path
7,A004.sol,0xcae22909c9dbc37c2f6c2780d1f58decd5d3b1fd,{39},None,{41},None,True,{41},True,post_claude_results_train_Integer_Overflow_and...
11,Distribute.sol,0x84004ca3b679b94d5c27c59d710e8201068bb93b,{13},None,{11},None,True,{14},True,post_claude_results_train_Integer_Overflow_and...
13,ITSETH.sol,0xfb9ba5b45ffb009a369fdb5140fd470e7bdcdd66,{62},None,{57},None,True,{62},True,post_claude_results_train_Integer_Overflow_and...
14,myPreICO.sol,0x35e44051799bd7cea091c4fad2cba9b37e364c5b,{30},None,{25},None,True,"{50, 51, 30}",True,post_claude_results_train_Integer_Overflow_and...
17,DungeonCoreBeta.sol,0x141766882733cafa9033e8707548fdcac908db22,{540},None,{537},None,True,"{481, 482, 483, 709, 710, 711, 537, 463, 466, ...",True,post_claude_results_train_Integer_Overflow_and...
19,EDCoreVersion1.sol,0xf7ed56c1ac4d038e367a987258b86fc883b960a1,{1070},None,{1067},None,True,None,False,None
25,UserfeedsClaimWithConfigurableTokenMultiTransf...,0xecbed48098c4f25a16195c45ddf5fd736e28b14b,{131},None,{130},None,True,None,False,None
40,ESmart.sol,0x84cd9cf60bcb44f7bab8b75e6f03614c2c3b22b7,{266},None,{262},None,True,{262},True,post_claude_results_train_Integer_Overflow_and...
41,AdoreNetwork.sol,0x663a9eba94dad8ca08917549ceb10565dffd5351,{187},None,{190},None,True,"{140, 141, 175, 177, 180, 186, 187}",True,post_claude_results_train_Integer_Overflow_and...
46,CrystalDeposit.sol,0xcb8361f6ea2c93a39ffd9e8357ed6ccd48331dea,{316},None,{314},None,True,None,False,None


### Validate and correct the claude output

In [185]:
import pandas as pd
import pickle
import os
import json
import random
import numpy as np
import anthropic
import warnings
import time
import fnmatch
import re

with warnings.catch_warnings():
    warnings.simplefilter("ignore")

cwd = os.getcwd()

train = pd.read_parquet(f'{cwd}\\Vulnerable-Verified-Smart-Contracts\\train.parquet')
test = pd.read_parquet(f'{cwd}\\Vulnerable-Verified-Smart-Contracts\\test.parquet')
validation = pd.read_parquet(f'{cwd}\\Vulnerable-Verified-Smart-Contracts\\validation.parquet')

def add_new_columns(dataset):
    vulns = ['IOU', 'TOD', 'RE', 'UcC', 'UpS', 'TD', 'TO', 'DC', 'NC', 'FE']
    tools = ['soldetector', 'slither', 'oyente', 'smartcheck']

    # Initialize new columns
    for vuln in vulns:
        dataset[f'{vuln}'] = "Undetected"
        for tool in tools:
            dataset[f'{vuln}_{tool}'] = "Undetected"
            
    for index, row in dataset.iterrows():
        overlapping = row['overlapping']
    
        # Parse JSON string to Python dictionary
        data = json.loads(overlapping)
    
        vulns = data.keys()
        for vuln in vulns:
            info_dict_list = data[vuln][0]
            dataset.at[index, f'{vuln}'] = "Detected"
            for info_dict in info_dict_list:
                # print(info_dict)
                try:
                    lines = str(info_dict['lines'])
                except KeyError as e:
                    lines = "[" + str(info_dict['line']) + "]"
                tool = info_dict['tool']
                dataset.at[index, f'{vuln}_{tool}'] = lines

def check_for_improper_detections(dataset):
    vulns = ['IOU', 'TOD', 'RE', 'UcC', 'UpS', 'TD', 'TO', 'DC', 'NC', 'FE']
    tools = ['soldetector', 'slither', 'oyente', 'smartcheck']
    result = dict()
    
    for vuln in vulns:
        vuln_df = dataset[dataset[vuln] == "Detected"]

        if len(vuln_df) > 0:
            # print("Vulnerability: ", vuln, "Number of samples: ", len(vuln_df))
            df = pd.DataFrame(columns=["file_path", "contract_address", "soldetector", "slither", "oyente", "smartcheck", "need_intervention"])
            # Set all columns except 'need_intervention' to 'object' type
            df[df.columns.difference(['need_intervention'])] = df[df.columns.difference(['need_intervention'])].astype('object')
            
            row_index = 0
            for index, row in vuln_df.iterrows():
                df.loc[row_index] = [row['file_path'], row['contract_address'], None, None, None, None, False]
                vul_lines_set_for_all_tool = []
                need_intervention = False
                for tool in tools:
                    vul_lines_set_for_tool = set()
                    if row[f'{vuln}_{tool}'] != "Undetected":
                        line_list = eval(row[f'{vuln}_{tool}'])
                        vul_lines_set_for_tool.update(line_list)
                        vul_lines_set_for_all_tool.append(vul_lines_set_for_tool)
                        if len(vul_lines_set_for_tool) > 0:
                            df.loc[row_index, tool] = str(vul_lines_set_for_tool)
                for i in range(len(vul_lines_set_for_all_tool)):
                    for j in range(len(vul_lines_set_for_all_tool)):
                        if i == j:
                            continue
                        if vul_lines_set_for_all_tool[i] != vul_lines_set_for_all_tool[j]:
                            need_intervention = True
                df.loc[row_index, "need_intervention"] = need_intervention
                row_index += 1
            result[vuln] = df
    return result

def count_need_intervention(result_dict):
    # print("Vulberability types: ", list(result_dict.keys()))
    # print("Number of vulnerability types: ", len(result_dict.keys()))
    count_dict = dict()
    count_sum = 0
    for key in result_dict.keys():
        df = result_dict[key]
        true_count = (df['need_intervention'] == True).sum()
        count_sum += true_count
        count_dict[key] = true_count
    return count_sum, count_dict

# Add new columns to the dataframes for further processing
add_new_columns(train)
add_new_columns(test)
add_new_columns(validation)

result_dict_train = check_for_improper_detections(train)
result_dict_test = check_for_improper_detections(test)
result_dict_validation = check_for_improper_detections(validation)

count_train, count_dict_train = count_need_intervention(result_dict_train)
count_test, count_dict_test = count_need_intervention(result_dict_test)
count_validation, count_dict_validation = count_need_intervention(result_dict_validation)

print("Train dataset\n===============================\n")
print("Number of contracts needing intervention: ", count_train)
print("Number of contracts needing intervention by vuln. type: ", count_dict_train)

print("\n\nTest dataset\n===============================\n")
print("Number of contracts needing intervention: ", count_test)
print("Number of contracts needing intervention by vuln. type: ", count_dict_test)

print("\n\nValidation dataset\n===============================")
print("Number of contracts needing intervention: ", count_validation)
print("Number of contracts needing intervention by vuln. type: ", count_dict_validation)

vulns_mapping = {'IOU': "Integer_Overflow_and_Underflow", 'TOD': "Transaction_Order_Dependency", 'RE': "Reentrancy", 'UcC': "Unchecked_Call", 'UpS': "Unprotected_Suicide", 'TD': "Timestamp_Dependency", 'TO': "TxOrigin", 'DC': "DelegateCall", 'NC': "Nested_Call", 'FE': "Frozen_Ether"}
reverse_vulns_mapping = {v: k for k, v in vulns_mapping.items()}
print(reverse_vulns_mapping)

Train dataset

Number of contracts needing intervention:  267
Number of contracts needing intervention by vuln. type:  {'IOU': 30, 'TOD': 1, 'RE': 56, 'UcC': 0, 'UpS': 64, 'TD': 5, 'TO': 0, 'DC': 9, 'NC': 100, 'FE': 2}


Test dataset

Number of contracts needing intervention:  16
Number of contracts needing intervention by vuln. type:  {'IOU': 2, 'TOD': 0, 'RE': 2, 'UcC': 0, 'UpS': 3, 'TD': 1, 'DC': 1, 'NC': 7}


Validation dataset
Number of contracts needing intervention:  18
Number of contracts needing intervention by vuln. type:  {'IOU': 0, 'TOD': 0, 'RE': 1, 'UcC': 0, 'UpS': 5, 'TD': 0, 'TO': 0, 'DC': 1, 'NC': 11}
{'Integer_Overflow_and_Underflow': 'IOU', 'Transaction_Order_Dependency': 'TOD', 'Reentrancy': 'RE', 'Unchecked_Call': 'UcC', 'Unprotected_Suicide': 'UpS', 'Timestamp_Dependency': 'TD', 'TxOrigin': 'TO', 'DelegateCall': 'DC', 'Nested_Call': 'NC', 'Frozen_Ether': 'FE'}


In [186]:
from rich.console import Console
from rich.syntax import Syntax
from IPython.display import Markdown

def check_code_for_correctness(solidity_code, highlight_lines):
    console = Console()
    
    # Create a Syntax object with highlighting
    syntax = Syntax(solidity_code, "solidity", line_numbers=True, highlight_lines=highlight_lines)
    
    # Print formatted code with highlights
    console.print(syntax)


indices_list = []
def get_indices_list(curr_df):
    global indices_list
    indices_list = curr_df.index.tolist()

def get_claude_output(pkl_file_path, file_path, contract_address):
    with open(pkl_file_path, 'rb') as file:
        data = pickle.load(file)
        for item in data:
            if item['file_path'] == file_path and item['contract_address'] == contract_address:
                return item['response'][-1].text

def check_one_by_one(curr_df, curr_df_name):
    curr_index = indices_list.pop()
    file_path = curr_df.loc[curr_index, 'file_path']
    contract_address = curr_df.loc[curr_index, 'contract_address']
    print("index: ", curr_index)
    print("file_path: ", file_path)
    print("contract_address: ", contract_address)
    
    soldetector = curr_df.loc[curr_index, 'soldetector']
    slither = curr_df.loc[curr_index, 'slither']
    oyente = curr_df.loc[curr_index, 'oyente']
    smartcheck = curr_df.loc[curr_index, 'smartcheck']
    claude = curr_df.loc[curr_index, 'claude']
    claude_complete = curr_df.loc[curr_index, 'claude_complete']

    if isinstance(soldetector, str):
        print("soldetector: ", soldetector)
        soldetector = eval(soldetector)

    if isinstance(slither, str):
        print("slither: ", slither)
        slither = eval(slither)

    if isinstance(oyente, str):
        print("oyente: ", oyente)
        oyente = eval(oyente)

    if isinstance(smartcheck, str):
        print("smartcheck: ", smartcheck)
        smartcheck = eval(smartcheck)

    if isinstance(claude, str):
        print("claude: ", claude)
    
        source_code = None
        if "train" in curr_df_name:
            source_code = train[(train['file_path'] == file_path) & (train['contract_address'] == contract_address)].iloc[0]['source_code']
        elif "test" in curr_df_name:
            source_code = test[(test['file_path'] == file_path) & (test['contract_address'] == contract_address)].iloc[0]['source_code']
        elif "validation" in curr_df_name:
            source_code = validation[(validation['file_path'] == file_path) & (validation['contract_address'] == contract_address)].iloc[0]['source_code']
        else:
            print("Error while getting the source code", file_path, contract_address)
        check_code_for_correctness(source_code, eval(claude))
        print("\nclaude output: \n")
        display(Markdown(get_claude_output(curr_df.loc[curr_index, 'pkl_file_path'], file_path, contract_address)))
    else:
        print("claude column is not a string", file_path, contract_address)


In [357]:
curr_df_name = "validation_UpS.csv"
df = pd.read_csv(curr_df_name)
print(len(df))
df = df[(df['need_intervention'] == True) & (df['claude_complete'] == True)]
get_indices_list(df)
print(len(df))

7
3


In [363]:
check_one_by_one(df, curr_df_name)

index:  0
file_path:  Auctionify.sol
contract_address:  0x3a6984181f49c9ef64ada32373e736db0c9f1078
soldetector:  {178}
slither:  {173, 174, 175, 176, 177, 178, 179, 180}
claude:  {173}


    1 pragma solidity ^0.4.22;                                                                                     
    2                                                                                                              
    3 /// @title Auctionify, A platform to auction stuff, using ethereum                                           
    4 /// @author Auctionify.xyz                                                                                   
    5 /// @notice This is the stand alone version of the auction                                                   
    6 /// // @dev All function calls are currently implement without side effects                                  
    7 contract Auctionify {                                                                                        
    8     // Parameters of the auction.                                                                            
    9     // Time is absolute unix timestamps                                                                      
   10                                                                                                              
   11     address public beneficiary;                                                                              
   12     uint public auctionEnd;                                                                                  
   13     string public auctionTitle;                                                                              
   14     string public auctionDescription;                                                                        
   15     uint public minimumBid;                                                                                  
   16                                                                                                              
   17     // Escrow                                                                                                
   18     address public escrowModerator;                                                                          
   19     //bool public escrowEnabled;                                                                             
   20                                                                                                              
   21     // Current state of the auction.                                                                         
   22     address public highestBidder;                                                                            
   23                                                                                                              
   24     // List of all the bids                                                                                  
   25     mapping(address => uint) public bids;                                                                    
   26                                                                                                              
   27     // State of the Auction                                                                                  
   28     enum AuctionStates { Started, Ongoing, Ended }                                                           
   29     AuctionStates public auctionState;                                                                       
   30                                                                                                              
   31                                                                                                              
   32     //modifiers                                                                                              
   33     modifier auctionNotEnded()                                                                               
   34     {                                                                                                        
   35         // Revert the call if the bidding         


claude output: 



To analyze the contract for Unprotected Suicide vulnerabilities, I need to identify where the `selfdestruct` function is being used without proper access controls.

In this contract, the `cleanUpAfterYourself()` function contains calls to `selfdestruct()`:

```solidity
function cleanUpAfterYourself() public {
  require(auctionState == AuctionStates.Ended, "Auction is not ended.");
    if (escrowModerator != address(0)) {
      selfdestruct(escrowModerator);
    } else {
      selfdestruct(beneficiary); //save blockchain space, save lives
    }
}
```

This function is declared as `public`, meaning anyone can call it. While there is a check to ensure the auction has ended (`require(auctionState == AuctionStates.Ended)`), there is no access control to restrict who can trigger the self-destruct operation.

The auction state is set to `Ended` in the `endAuction()` function, which does have access controls through the `onlyHighestBidderOrEscrow` modifier. However, once the auction state is set to `Ended`, absolutely anyone can call `cleanUpAfterYourself()` to destroy the contract.

This is dangerous because:
1. It allows any user to destroy the contract after the auction ends
2. Any remaining funds in the contract will be forcibly sent to either the escrow moderator or beneficiary
3. It could disrupt contract state and functionality if called unexpectedly

The function should be protected with appropriate access controls similar to those used in the `endAuction()` function.

VULNERABLE_LINES: [function cleanUpAfterYourself() public {]

## Clean the claude incomplete vulnerabilities

In [181]:
dataset_files = ["test_DC.csv",
"test_IOU.csv",
"test_NC.csv",
"test_RE.csv",
"test_TD.csv",
"test_TOD.csv",
"test_UcC.csv",
"test_UpS.csv",
"train_DC.csv",
"train_FE.csv",
"train_IOU.csv",
"train_NC.csv",
"train_RE.csv",
"train_TD.csv",
"train_TO.csv",
"train_TOD.csv",
"train_UcC.csv",
"train_UpS.csv",
"validation_DC.csv",
"validation_IOU.csv",
"validation_NC.csv",
"validation_RE.csv",
"validation_TD.csv",
"validation_TO.csv",
"validation_TOD.csv",
"validation_UcC.csv",
"validation_UpS.csv"]

all_vulns = 0
need_intervention = 0
claude_complete = 0
claude_incomplete = 0

# List to hold the filtered rows
incomplete_rows = []

for dataset_file in dataset_files:
    df = pd.read_csv(dataset_file)
    all_vulns += len(df)
    vuln_label = (dataset_file.split("_")[1]).split(".csv")[0]
    # print(vuln_label)
    df['vuln'] = vuln_label
    need_intervention += (df['need_intervention'] == True).sum()
    claude_complete += ((df['need_intervention'] == True) & (df['claude_complete'] == True)).sum()
    claude_incomplete += ((df['need_intervention'] == True) & (df['claude_complete'] == False)).sum()

    # Select rows meeting the condition and append to the list
    condition = (df['need_intervention'] == True) & (df['claude_complete'] == False)
    incomplete_rows.append(df[condition])

# Concatenate all filtered rows into a single DataFrame
incomplete_df = pd.concat(incomplete_rows, ignore_index=True)
incomplete_df.to_csv("claude_incomplete_dataset.csv")

print("all_vulns: ", all_vulns)
print("need_intervention: ", need_intervention)
print("claude_complete: ", claude_complete)
print("claude_incomplete: ", claude_incomplete)

all_vulns:  822
need_intervention:  301
claude_complete:  259
claude_incomplete:  42


In [103]:
incomplete_df.head()

,Unnamed: 0,file_path,contract_address,soldetector,slither,oyente,smartcheck,need_intervention,claude,claude_complete,pkl_file_path,vuln
0,0,Pair.sol,0x0b3ded1b54e89f8899482cfccf515d525835bc2b,{472},"{472, 473}",NaN,NaN,True,NaN,False,NaN,DC
1,0,GetsBurned.sol,0x5f15d2d4c60e229586a5bdfe2eec1981f92845c1,{9},"{8, 9, 10}",NaN,NaN,True,{8},False,post_claude_results_test_Unprotected_Suicide.pkl,UpS
2,9,MoonbaseNFTAirdrop.sol,0x514aade50d0a679afd172720a7b6fbaccccb64ee,{826},"{832, 833, 826, 827, 828, 829, 830, 831}",NaN,NaN,True,set(),False,post_claude_results_train_DelegateCall.pkl,DC
3,16,RibbonThetaVault.sol,0x00a62ee3d2998f67cc202990b792573961d282e6,{907},"{907, 908, 909, 910, 911, 912, 913, 914, 915, ...",NaN,NaN,True,Not_Found,False,post_claude_results_train_DelegateCall.pkl,DC
4,18,contracts/pools/loss/MasterPool.sol,0xb9112feef2054acc4066c40e8c2784fa3e9d032f,{392},"{392, 393, 394}",NaN,NaN,True,set(),False,post_claude_results_train_DelegateCall.pkl,DC


### After mending the entries that the pkl file name is none

In [114]:
import pandas as pd
df = pd.read_csv("claude_incomplete_dataset.csv")
df.head()

,Unnamed: 0.1,Unnamed: 0,file_path,contract_address,soldetector,slither,oyente,smartcheck,need_intervention,claude,claude_complete,pkl_file_path,vuln
0,0,0,Pair.sol,0x0b3ded1b54e89f8899482cfccf515d525835bc2b,{472},"{472, 473}",NaN,NaN,True,"{472, 473, 488, 489, 535, 536}",True,NaN,DC
1,1,0,GetsBurned.sol,0x5f15d2d4c60e229586a5bdfe2eec1981f92845c1,{9},"{8, 9, 10}",NaN,NaN,True,"{8, 9}",True,post_claude_results_test_Unprotected_Suicide.pkl,UpS
2,2,9,MoonbaseNFTAirdrop.sol,0x514aade50d0a679afd172720a7b6fbaccccb64ee,{826},"{832, 833, 826, 827, 828, 829, 830, 831}",NaN,NaN,True,"{826, 827, 828, 829, 830, 831, 832, 833}",True,post_claude_results_train_DelegateCall.pkl,DC
3,3,16,RibbonThetaVault.sol,0x00a62ee3d2998f67cc202990b792573961d282e6,{907},"{907, 908, 909, 910, 911, 912, 913, 914, 915, ...",NaN,NaN,True,"{456, 871, 872, 873, 874, 875, 876, 877, 878, ...",True,post_claude_results_train_DelegateCall.pkl,DC
4,4,18,contracts/pools/loss/MasterPool.sol,0xb9112feef2054acc4066c40e8c2784fa3e9d032f,{392},"{392, 393, 394}",NaN,NaN,True,"{392, 393, 394}",True,post_claude_results_train_DelegateCall.pkl,DC


In [116]:
print((df['claude_complete'] == True).sum())
print((df['claude_complete'] == False).sum())

42
0


In [110]:
from rich.console import Console
from rich.syntax import Syntax
from IPython.display import Markdown

def check_code_for_correctness(solidity_code):
    console = Console()
    
    # Create a Syntax object with highlighting
    syntax = Syntax(solidity_code, "solidity", line_numbers=True)
    
    # Print formatted code with highlights
    console.print(syntax)


indices_list = []
def get_indices_list(curr_df):
    global indices_list
    indices_list = curr_df.index.tolist()

def get_claude_output(pkl_file_path, file_path, contract_address):
    with open(pkl_file_path, 'rb') as file:
        data = pickle.load(file)
        for item in data:
            if item['file_path'] == file_path and item['contract_address'] == contract_address:
                return item['response'][-1].text

def check_one_by_one(curr_df):
    curr_index = indices_list.pop()
    file_path = curr_df.loc[curr_index, 'file_path']
    contract_address = curr_df.loc[curr_index, 'contract_address']
    print("index: ", curr_index)
    print("file_path: ", file_path)
    print("contract_address: ", contract_address)
    
    soldetector = curr_df.loc[curr_index, 'soldetector']
    slither = curr_df.loc[curr_index, 'slither']
    oyente = curr_df.loc[curr_index, 'oyente']
    smartcheck = curr_df.loc[curr_index, 'smartcheck']
    claude = curr_df.loc[curr_index, 'claude']
    claude_complete = curr_df.loc[curr_index, 'claude_complete']

    if isinstance(soldetector, str):
        print("soldetector: ", soldetector)
        soldetector = eval(soldetector)

    if isinstance(slither, str):
        print("slither: ", slither)
        slither = eval(slither)

    if isinstance(oyente, str):
        print("oyente: ", oyente)
        oyente = eval(oyente)

    if isinstance(smartcheck, str):
        print("smartcheck: ", smartcheck)
        smartcheck = eval(smartcheck)

    if isinstance(claude, str):
        print("claude: ", claude)

        pkl_file_path = curr_df.loc[curr_index, 'pkl_file_path']
        source_code = None
        if "train" in pkl_file_path:
            source_code = train[(train['file_path'] == file_path) & (train['contract_address'] == contract_address)].iloc[0]['source_code']
        elif "test" in pkl_file_path:
            source_code = test[(test['file_path'] == file_path) & (test['contract_address'] == contract_address)].iloc[0]['source_code']
        elif "validation" in pkl_file_path:
            source_code = validation[(validation['file_path'] == file_path) & (validation['contract_address'] == contract_address)].iloc[0]['source_code']
        else:
            print("Error while getting the source code", file_path, contract_address)
        check_code_for_correctness(source_code)
        print("\nclaude output: \n")
        display(Markdown(get_claude_output(curr_df.loc[curr_index, 'pkl_file_path'], file_path, contract_address)))
    else:
        print("claude column is not a string", file_path, contract_address)


In [112]:
curr_df_name = "claude_incomplete_dataset.csv"
df = pd.read_csv(curr_df_name)
df = df[(df['need_intervention'] == True) & (df['claude_complete'] == False)]
get_indices_list(df)
print(len(df))

42


In [81]:
check_one_by_one(df)

index:  40
file_path:  GetsBurned.sol
contract_address:  0x2f976a5382c0c6214b3d8fe6bdc5379edd35d9d4
soldetector:  {8}
slither:  {8, 9, 6, 7}
claude:  {6}


   1 pragma solidity ^0.4.0;                                                                                       
   2 contract GetsBurned {                                                                                         
   3     function () payable public {                                                                              
   4     }                                                                                                         
   5                                                                                                               
   6     function BurnMe() public {                                                                                
   7         // Selfdestruct and send eth to self,                                                                 
   8         selfdestruct(address(this));                                                                          
   9     }                                                                                                         
  10 }                                                                                                             


claude output: 



After analyzing the code, I need to identify the Unprotected Suicide vulnerability.

An Unprotected Suicide vulnerability occurs when a contract can be destroyed via `selfdestruct` without proper access controls, allowing anyone to terminate the contract and potentially cause loss of funds or service disruption.

In this contract:
1. The `BurnMe()` function is publicly accessible with no access controls
2. This function calls `selfdestruct`, which destroys the contract
3. Anyone can call this function to terminate the contract at any time
4. The funds are sent to `address(this)` (the contract itself), which means the funds will be lost forever when the contract is destroyed

The main issue is that there's no protection on who can call the self-destruct functionality - no owner check, no authorization mechanism, and no other form of access control.

The vulnerable parts are both the function declaration (which lacks access controls) and the selfdestruct call itself.

VULNERABLE_LINES: [function BurnMe() public {, selfdestruct(address(this));}]

# Merge all cleaned datasets to one

In [14]:
import pandas as pd
import pickle
import os
import json
import random
import numpy as np
import anthropic
import warnings
import time
import fnmatch
import re

with warnings.catch_warnings():
    warnings.simplefilter("ignore")

cwd = os.getcwd()

train = pd.read_parquet(f'{cwd}\\Vulnerable-Verified-Smart-Contracts\\train.parquet')
test = pd.read_parquet(f'{cwd}\\Vulnerable-Verified-Smart-Contracts\\test.parquet')
validation = pd.read_parquet(f'{cwd}\\Vulnerable-Verified-Smart-Contracts\\validation.parquet')

def add_new_columns(dataset):
    vulns = ['IOU', 'TOD', 'RE', 'UcC', 'UpS', 'TD', 'TO', 'DC', 'NC', 'FE']
    tools = ['soldetector', 'slither', 'oyente', 'smartcheck']

    # Initialize new columns
    for vuln in vulns:
        dataset[f'{vuln}'] = "Undetected"
        for tool in tools:
            dataset[f'{vuln}_{tool}'] = "Undetected"
            
    for index, row in dataset.iterrows():
        overlapping = row['overlapping']
    
        # Parse JSON string to Python dictionary
        data = json.loads(overlapping)
    
        vulns = data.keys()
        for vuln in vulns:
            info_dict_list = data[vuln][0]
            dataset.at[index, f'{vuln}'] = "Detected"
            for info_dict in info_dict_list:
                # print(info_dict)
                try:
                    lines = str(info_dict['lines'])
                except KeyError as e:
                    lines = "[" + str(info_dict['line']) + "]"
                tool = info_dict['tool']
                dataset.at[index, f'{vuln}_{tool}'] = lines

def check_for_improper_detections(dataset):
    vulns = ['IOU', 'TOD', 'RE', 'UcC', 'UpS', 'TD', 'TO', 'DC', 'NC', 'FE']
    tools = ['soldetector', 'slither', 'oyente', 'smartcheck']
    result = dict()
    
    for vuln in vulns:
        vuln_df = dataset[dataset[vuln] == "Detected"]

        if len(vuln_df) > 0:
            # print("Vulnerability: ", vuln, "Number of samples: ", len(vuln_df))
            df = pd.DataFrame(columns=["file_path", "contract_address", "soldetector", "slither", "oyente", "smartcheck", "need_intervention"])
            # Set all columns except 'need_intervention' to 'object' type
            df[df.columns.difference(['need_intervention'])] = df[df.columns.difference(['need_intervention'])].astype('object')
            
            row_index = 0
            for index, row in vuln_df.iterrows():
                df.loc[row_index] = [row['file_path'], row['contract_address'], None, None, None, None, False]
                vul_lines_set_for_all_tool = []
                need_intervention = False
                for tool in tools:
                    vul_lines_set_for_tool = set()
                    if row[f'{vuln}_{tool}'] != "Undetected":
                        line_list = eval(row[f'{vuln}_{tool}'])
                        vul_lines_set_for_tool.update(line_list)
                        vul_lines_set_for_all_tool.append(vul_lines_set_for_tool)
                        if len(vul_lines_set_for_tool) > 0:
                            df.loc[row_index, tool] = str(vul_lines_set_for_tool)
                for i in range(len(vul_lines_set_for_all_tool)):
                    for j in range(len(vul_lines_set_for_all_tool)):
                        if i == j:
                            continue
                        if vul_lines_set_for_all_tool[i] != vul_lines_set_for_all_tool[j]:
                            need_intervention = True
                df.loc[row_index, "need_intervention"] = need_intervention
                row_index += 1
            result[vuln] = df
    return result

# def count_need_intervention(result_dict):
#     # print("Vulberability types: ", list(result_dict.keys()))
#     # print("Number of vulnerability types: ", len(result_dict.keys()))
#     count_dict = dict()
#     count_sum = 0
#     for key in result_dict.keys():
#         df = result_dict[key]
#         true_count = (df['need_intervention'] == True).sum()
#         count_sum += true_count
#         count_dict[key] = true_count
#     return count_sum, count_dict

# Add new columns to the dataframes for further processing
add_new_columns(train)
add_new_columns(test)
add_new_columns(validation)

result_dict_train = check_for_improper_detections(train)
result_dict_test = check_for_improper_detections(test)
result_dict_validation = check_for_improper_detections(validation)

# count_train, count_dict_train = count_need_intervention(result_dict_train)
# count_test, count_dict_test = count_need_intervention(result_dict_test)
# count_validation, count_dict_validation = count_need_intervention(result_dict_validation)

# print("Train dataset\n===============================\n")
# print("Number of contracts needing intervention: ", count_train)
# print("Number of contracts needing intervention by vuln. type: ", count_dict_train)

# print("\n\nTest dataset\n===============================\n")
# print("Number of contracts needing intervention: ", count_test)
# print("Number of contracts needing intervention by vuln. type: ", count_dict_test)

# print("\n\nValidation dataset\n===============================")
# print("Number of contracts needing intervention: ", count_validation)
# print("Number of contracts needing intervention by vuln. type: ", count_dict_validation)

# vulns_mapping = {'IOU': "Integer_Overflow_and_Underflow", 'TOD': "Transaction_Order_Dependency", 'RE': "Reentrancy", 'UcC': "Unchecked_Call", 'UpS': "Unprotected_Suicide", 'TD': "Timestamp_Dependency", 'TO': "TxOrigin", 'DC': "DelegateCall", 'NC': "Nested_Call", 'FE': "Frozen_Ether"}
# reverse_vulns_mapping = {v: k for k, v in vulns_mapping.items()}
# print(reverse_vulns_mapping)

In [15]:
vulns = ['IOU', 'TOD', 'RE', 'UcC', 'UpS', 'TD', 'TO', 'DC', 'NC', 'FE']

In [18]:
def add_cleaned_rows(df, df1):
    for df1_index, df1_row in df1.iterrows():
        condition = (df['file_path'] == df1_row['file_path']) & (df['contract_address'] == df1_row['contract_address']) & (df['vuln'] == df1_row['vuln'])
        matching_indexes = df.index[condition].tolist()
        if len(matching_indexes) == 1:
            match_index = matching_indexes[0]
            if df.loc[match_index, 'need_intervention'] == True:
                df.loc[match_index, 'claude'] = df1_row['claude']
                df.loc[match_index, 'claude_complete'] = df1_row['claude_complete']
                df.loc[match_index, 'final_line_numbers'] = df1_row['claude']
                df.loc[match_index, 'pkl_file_path'] = df1_row['pkl_file_path']
        else:
            if len(matching_indexes) > 1:
                print("Unexpected error: more than 1 matching index")
    return df

result_dicts = {"train": result_dict_train, "test": result_dict_test, "validation": result_dict_validation}
dfs = []
count = 0
for result_dict_key in list(result_dicts.keys()):
    result_dict = result_dicts[result_dict_key]
    for vuln in list(result_dict.keys()):
        df = result_dict[vuln]

        count += len(df)
        
        df['final_line_numbers'] = None
        df['claude'] = None
        df['claude_complete'] = False
        df['pkl_file_path'] = None
        df['vuln'] = vuln
        df['type'] = result_dict_key

        for index, row in df.iterrows():
            if row['need_intervention'] == False:
                soldetector = row['soldetector']
                slither = row['slither']
                oyente = row['oyente']
                smartcheck = row['smartcheck']
                final_line_numbers = None
                if slither is not None and isinstance(slither, str):
                    final_line_numbers = slither
                elif soldetector is not None and isinstance(soldetector, str):
                    final_line_numbers = soldetector
                elif oyente is not None and isinstance(oyente, str):
                    final_line_numbers = oyente
                elif smartcheck is not None and isinstance(smartcheck, str):
                    final_line_numbers = smartcheck
                else:
                    print("Unexpected error: all detectors results are empty")
                df.loc[index, 'final_line_numbers'] = str(eval(final_line_numbers))

        if os.path.exists(f"{result_dict_key}_{vuln}_curated.csv"):
            # print(f"{result_dict_key}_{vuln}_curated.csv exists")
            df1 = pd.read_csv(f"{result_dict_key}_{vuln}_curated.csv")
            for df1_index, df1_row in df1.iterrows():
                condition = (df['file_path'] == df1_row['file_path']) & (df['contract_address'] == df1_row['contract_address']) & (df['vuln'] == vuln)
                matching_indexes = df.index[condition].tolist()
                if len(matching_indexes) == 1:
                    match_index = matching_indexes[0]
                    if df.loc[match_index, 'need_intervention'] == True:
                        df.loc[match_index, 'claude'] = df1_row['claude']
                        df.loc[match_index, 'claude_complete'] = df1_row['claude_complete']
                        df.loc[match_index, 'final_line_numbers'] = df1_row['claude']
                        df.loc[match_index, 'pkl_file_path'] = df1_row['pkl_file_path']
                else:
                    if len(matching_indexes) > 1:
                        print("Unexpected error: more than 1 matching index")
        result_dict[vuln] = df
        dfs.append(df)

merged_df = pd.concat(dfs, ignore_index=True)

df = pd.read_csv("claude_incomplete_dataset.csv")
df.drop(['Unnamed: 0.1', 'Unnamed: 0'], axis=1, inplace=True)
final_per_vuln_df = add_cleaned_rows(merged_df, df)
final_per_vuln_df.to_csv("processed/vulnerable_verified_smart_contracts_dataset_with_line_numbers_per_vulnerability.csv")
final_per_vuln_df

,file_path,contract_address,soldetector,slither,oyente,smartcheck,need_intervention,final_line_numbers,claude,claude_complete,pkl_file_path,vuln,type
0,Gift.sol,0xb91a6c5c6362b10db6440d690e5391bb1eabe591,{14},None,{14},None,False,{14},None,False,None,IOU,train
1,TokenERC20.sol,0xd70e66775a74a9aedeac6e28313e1033cd726552,{441},None,{441},None,False,{441},None,False,None,IOU,train
2,Vault.sol,0x598ab825d607ace3b00d8714c0a141c7ae2e6822,{270},None,{270},None,False,{270},None,False,None,IOU,train
3,Sandstone.sol,0xae3bf0f077ed66dda9fb1b5475942c919ef3bb0d,{14},None,{14},None,False,{14},None,False,None,IOU,train
4,bet1000_001eth.sol,0xab3b0810b839f49c2d619b55f196aae764422e22,{127},None,{127},None,False,{127},None,False,None,IOU,train
...,...,...,...,...,...,...,...,...,...,...,...,...,...
817,SkyWay.sol,0x8235cfc3e7d9500b16cdf23ea66979f2d5ab3c74,{105},{107},None,{105},True,"{105, 107}","{105, 107}",True,NaN,NC,validation
818,BulkSender.sol,0xcf8b99880a5ae950fe2a0b949e69a2e3e9e73c3e,{86},{90},None,{86},True,"{90, 86}","{90, 86}",True,post_claude_results_validation_Nested_Call.pkl,NC,validation
819,UnilotTailEther.sol,0x468480af8817914062e1d198eb238e5087896f86,{372},{374},None,{372},True,"{446, 442, 374, 372}","{446, 442, 374, 372}",True,post_claude_results_validation_Nested_Call.pkl,NC,validation
820,MultiTransfer.sol,0xb47ba2f58795f87c71403fbcae57a3aee902e175,{33},{34},None,{33},True,"{34, 33}","{34, 33}",True,post_claude_results_validation_Nested_Call.pkl,NC,validation


In [19]:
import pandas as pd

def concat_final_line_numbers(series):
    """
    Given a Series of string representations of lists or sets,
    evaluate each and combine all items into a set.
    """
    combined = set()
    for item in series:
        if isinstance(item, str):
            try:
                # Evaluate the string to convert it into a Python object.
                evaluated = eval(item)
                # If evaluated is iterable (but not a string), update the set.
                if hasattr(evaluated, '__iter__') and not isinstance(evaluated, str):
                    combined.update(evaluated)
                else:
                    combined.add(evaluated)
            except Exception as e:
                print(f"Error evaluating {item}: {e}")
        elif pd.notna(item):
            try:
                combined.update(item)
            except Exception:
                combined.add(item)
    return str(combined)

# Read the CSV file (update the path if needed)
df = pd.read_csv('processed/vulnerable_verified_smart_contracts_dataset_with_line_numbers_per_vulnerability.csv')

# Group by the compound key and aggregate the columns.
# For 'vulns' and 'type', join unique values separated by a comma.
grouped = df.groupby(['file_path', 'contract_address'], as_index=False).agg({
    'final_line_numbers': concat_final_line_numbers,
    'vuln': lambda x: ','.join(x.dropna().unique()),
    'type': lambda x: ','.join(x.dropna().unique())
})

grouped['source_code'] = None
for index, row in grouped.iterrows():
    source_code = None
    if row['type'] == 'train':
        source_code = train[(train['file_path'] == row['file_path']) & (train['contract_address'] == row['contract_address'])].iloc[0]['source_code']
    elif row['type'] == 'test':
        source_code = test[(test['file_path'] == row['file_path']) & (test['contract_address'] == row['contract_address'])].iloc[0]['source_code']
    elif row['type'] == 'validation':
        source_code = validation[(validation['file_path'] == row['file_path']) & (validation['contract_address'] == row['contract_address'])].iloc[0]['source_code']
    else:
        print("Error while getting the source code", file_path, contract_address)
    grouped.loc[index, 'source_code'] = source_code
        
grouped.to_csv('processed/vulnerable_verified_smart_contracts_dataset_with_line_numbers.csv', index=False)

In [20]:
grouped

,file_path,contract_address,final_line_numbers,vuln,type,source_code
0,@c-layer/common/contracts/core/Core.sol,0x2a903c2f657803a2e614c42672247366d757ab34,{55},DC,train,pragma solidity ^0.6.0;\r\n\r\n\r\n\r\n\r\n\r\...
1,A004.sol,0xcae22909c9dbc37c2f6c2780d1f58decd5d3b1fd,"{41, 46}","IOU,NC",train,pragma solidity ^0.4.25;\r\n\r\n\r\n\r\n/**\r\...
2,ABIO_preICO.sol,0x6d84769b1e287a27f282a938c8110b22714dbf78,{201},TD,train,pragma solidity ^0.4.24;\r\ncontract Ownable{\...
3,ACCURAL_DEPOSIT.sol,0x4320e6f8c05b27ab4707cd1f6d5ce6f3e4b3a5a1,"{49, 47}",RE,train,pragma solidity ^0.4.19;\r\n\r\ncontract ACCUR...
4,ACL.sol,0x96f041b96708813b1d789606926c524e78543664,{252},DC,train,//File: contracts/acl/IACL.sol\r\npragma solid...
...,...,...,...,...,...,...
604,ultra_bank.sol,0x094a5e6e395212bfc6c773b2210409eba4ca19d7,"{24, 20, 22}","RE,TD",train,pragma solidity ^0.4.25;\r\n\r\ncontract ultra...
605,we_play.sol,0x6df766e4b524aa1d4b4b9405994ca69dc161da3f,{34},TOD,test,pragma solidity ^0.4.25;\r\n\r\ncontract we_pl...
606,xcat.sol,0xb8aa8971e9201d183d1dadf5acc5c3f6b3076bc0,"{32, 25, 41, 30, 31}","IOU,UpS,TD",train,pragma solidity ^0.4.18;\r\n\r\ncontract HTLC ...
607,xyphar.sol,0x12359487aa7844fead6a20aca5bfae1d3196cedb,{46},NC,train,pragma solidity 0.4.24;\r\n\r\n\r\nlibrary Saf...
